In [ ]:
# Import package
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scanpy as sc
import igraph
import scvelo as scv
import loompy as lmp
import anndata as ad
import harmonypy as hm
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import bbknn

import celltypist
from celltypist import models
from celltypist.models import Model

# snRNA analysis

In [ ]:
adata_snraw=sc.read("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/snRNA_v1/snRNA_raw.h5ad")
adata_snrna=sc.read("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/snRNA_v1/snRNA_adata_01.h5ad")

In [ ]:
adata_snrna.obs['state']

In [ ]:
sc.pl.umap(adata_snrna[adata_snrna.obs['state'] == 'normal'], color='SLC7A5', title='Healthy')
sc.pl.umap(adata_snrna[adata_snrna.obs['state'] == 'MASLD'], color='SLC7A5', title='MASLD')

In [ ]:
adata_snraw=sc.read("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/snRNA_v1/snRNA_raw.h5ad")
sc.pp.normalize_total(adata_snraw, target_sum=1e4)
sc.pp.log1p(adata_snraw)
adata_snrna.raw = adata_snraw

In [ ]:
sc.pl.umap(
    adata_snrna,
    color='celltypist_liver',
    legend_loc='on data',
    legend_fontsize=6,
    frameon=False,
    title='Cell type prediction',
    size=10
)

In [ ]:
sc.pl.umap(adata_snrna,color='celltypist_liver')

In [ ]:
sc.pl.umap(adata_snrna,color='celltypist_liver')

In [ ]:
adata_snrna.obs

In [ ]:
sc.pl.umap(
    adata_snrna,
    color='cluster',
    frameon=False,
    #title='Lymphoid',
    size=3,
)

In [ ]:
adata_snrna.obs['celltypist_liver'].value_counts()

In [ ]:
sc.pl.umap(
    adata_snrna,
    color='PTPRC',
    frameon=False,
    #title='Lymphoid',
    size=3,
)

In [ ]:
sc.pl.umap(
    adata_snrna,
    color='MKI67',
    frameon=False,
    #title='Lymphoid',
    size=3,
)

In [ ]:
sc.pl.umap(
    adata_snrna,
    color=['VSIG4','CD5L'],
    frameon=False,
    #title='Lymphoid',
    size=3,
)

In [ ]:
adata_snrna.obs_names_make_unique()

In [ ]:
sc.tl.leiden(adata_snrna, resolution=1, key_added="cluster")
sc.pl.umap(adata_snrna, color=['cluster'])

In [ ]:
predictions_cluster = celltypist.annotate(adata_snrna, model='Healthy_Human_Liver.pkl', majority_voting=True, over_clustering=adata_snrna.obs["cluster"]
)
adata_snrna =predictions_cluster.to_adata()

In [ ]:
adata_snrna.obs['majority_voting'].value_counts()

In [ ]:
sc.pl.umap(
    adata_snrna,
    color='PTPRC',
    #legend_loc='on data',
    legend_fontsize=6,
    frameon=False,
    title='CD45+ prediction',
    size=10
)

In [ ]:
adata_snrna.write("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/snRNA_v1/snRNA_adata_01.h5ad")

In [ ]:
adata_snrna=sc.read("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/snRNA_v1/snRNA_adata_01.h5ad")
adata_snrna.obs['cluster'].value_counts()

In [ ]:
# 2. crosstab 생성
cluster_labels = adata_snrna.obs['celltypist_liver']
batch_labels = adata_snrna.obs['state']
crosstab = pd.crosstab(batch_labels, cluster_labels, normalize='index')

# 3. matplotlib barplot 그릴 때 palette 지정
crosstab.plot(
    kind='bar',
    stacked=True,
    figsize=(5, 10),
    #color=[palette_dict[col] for col in crosstab.columns]
)

plt.title('Major cell lineages in MASLD vs Normal')
plt.ylabel('Proportion')
plt.xlabel('state')
plt.legend(title='MASLD vs Normal', bbox_to_anchor=(1.05, 1), loc='upper left')
#plt.savefig("/home/jaehyunchoi/bystander_collaboration/IPF_data/analysis/figures/3publicatlas_celltypist_IPF_state_stackedbargraph.pdf", bbox_inches='tight', dpi=300)

In [ ]:
adata_snraw.obs['cluster'] = adata_snrna.obs['cluster']

# celltypist_macrophage sub cluster

In [ ]:
adata_snraw=sc.read("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/snRNA_v1/snRNA_raw.h5ad")
adata_snraw.obs['celltypist_liver'] = adata_snrna.obs['celltypist_liver']

In [ ]:
adata_snraw.obs['celltypist_liver']

In [ ]:
adata_snmono = adata_snraw[adata_snraw.obs['celltypist_liver'].isin(['Macrophages'])].copy()
adata_snmono

In [ ]:
adata_snmono.write("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/snRNA_v1/snRNA_adata_mono01_celltypistsub.h5ad")

In [ ]:
sc.pp.normalize_total(adata_snmono, target_sum=1e4)
sc.pp.log1p(adata_snmono)
adata_snmono.raw = adata_snmono
sc.pp.highly_variable_genes(adata_snmono, flavor='seurat', n_top_genes=2000)

In [ ]:
# Scale data
sc.pp.scale(adata_snmono, max_value=10)
# Run PCA
sc.tl.pca(adata_snmono, svd_solver='arpack')
print(adata_snmono.obsm['X_pca'][:5, :5])
sc.pl.pca(adata_snmono, color='sample')

import matplotlib.pyplot as plt
var_ratio = adata_snmono.uns['pca']['variance_ratio']
plt.plot(range(1, len(var_ratio) + 1), np.cumsum(var_ratio), marker='o')
plt.xlabel('Number of Principal Components')
plt.ylabel('Cumulative Explained Variance')
plt.title('PCA Cumulative Variance')
plt.grid(True)
plt.show()

In [ ]:
# t-sne 
sc.tl.tsne(adata_snmono, use_rep='X_pca', n_pcs=30)
# umap
sc.pp.neighbors(adata_snmono, n_pcs=30, use_rep='X_pca')
sc.tl.umap(adata_snmono)
sc.pl.umap(adata_snmono, color=['sample'])

In [ ]:
# Harmony로 batch effect 제거
harmony_out = hm.run_harmony(adata_snmono.obsm['X_pca'],adata_snmono.obs, 'sample')

# Harmony 결과를 저장
adata_snmono.obsm['X_pca_harmony'] = harmony_out.Z_corr.T

# UMAP을 사용한 시각화
sc.pp.neighbors(adata_snmono, use_rep='X_pca_harmony')  # Harmony 결과를 사용
sc.tl.umap(adata_snmono)

In [ ]:
adata_snmono.obs_names_make_unique()

In [ ]:
sc.pl.umap(adata_snmono,color='state')

In [ ]:
sc.tl.rank_genes_groups(
    adata_snmono,
    groupby="state",        # 비교할 obs 열
    groups=["MASLD"],     # 비교할 그룹
    reference="normal",     # 참조 그룹
    use_raw=True,
    method="wilcoxon"       # 또는 't-test', 'logreg'
)

In [ ]:
filtered = deg_df[
    (deg_df['logfoldchanges'] > 0.58) &
    (deg_df['pvals_adj'] < 0.05)
]
top20_filtered = filtered.sort_values(by='scores', ascending=False).head(50)
print(top20_filtered)

In [ ]:
deg_df['-log10(padj)'] = -np.log10(deg_df['pvals_adj'])
genes_of_interest = ["SLC7A5"]

plt.figure(figsize=(7, 6))
plt.scatter(
    deg_df['logfoldchanges'], 
    deg_df['-log10(padj)'], 
    c="grey", s=10, alpha=0.7
)

# 유의미한 cutoff 설정 (예: logFC > 1, padj < 0.05)
sig_up = (deg_df['logfoldchanges'] > 1) & (deg_df['pvals_adj'] < 0.05)
sig_down = (deg_df['logfoldchanges'] < -1) & (deg_df['pvals_adj'] < 0.05)

plt.scatter(deg_df.loc[sig_up, 'logfoldchanges'],
            deg_df.loc[sig_up, '-log10(padj)'],
            c="red", s=10, label="Upregulated")

plt.scatter(deg_df.loc[sig_down, 'logfoldchanges'],
            deg_df.loc[sig_down, '-log10(padj)'],
            c="blue", s=10, label="Downregulated")

# cutoff 선
plt.axvline(1, color="black", linestyle="--")
plt.axvline(-1, color="black", linestyle="--")
plt.axhline(-np.log10(0.05), color="black", linestyle="--")

plt.xlabel("log2 Fold Change")
plt.ylabel("-log10 Adjusted p-value")
plt.title("Volcano Plot: MASLD vs Normal")
plt.legend()

for gene in genes_of_interest:
    if gene in deg_df['names'].values:
        x = deg_df.loc[deg_df['names'] == gene, 'logfoldchanges'].values[0]
        y = deg_df.loc[deg_df['names'] == gene, '-log10(padj)'].values[0]
        plt.text(x, y, gene, fontsize=8, color="black")  # 텍스트 표시


plt.tight_layout()
plt.show()

In [ ]:
sc.pl.umap(
    adata_snmono,
    color='state',
    #legend_loc='on data',
    legend_fontsize=10,
    frameon=False,
    title='state',
    size=15
)

In [ ]:
adata_snmono.write("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/snRNA_v1/snRNA_adata_mono02_celltypistsub.h5ad")

In [ ]:
gene_sets = {
    "cDC1_genes": ['XCR1', 'CLNK', 'TACSTD2', 'CLEC9A', 'EGLN3', 'IDO1', 'DNASE1L3', 'CADM1', 'S100B', 'KIAA0226L', 'WDFY4', 'CCND1', 'DUSP4', 'P2RY14', 'C1orf54', 'NRARP', 'CPNE3', 'SLAMF7', 'GPR157', 'B3GNT7', 'FLT3', 'NET1', 'RGCC', 'SERPINF1', 'BATF3', 'NDRG2', 'SNX3', 'ADAM28', 'AIM2', 'IL18R1', 'GPR171', 'PTPRCAP', 'IRF8', 'CPVL', 'C12orf45', 'TPD52', 'SLC38A1', 'LMNA', 'CD59', 'C15orf48', 'NFATC2', 'NAV1', 'MAP2K6', 'PTMS', 'ARHGAP5', 'HLA-DPB1', 'CST7', 'LGALS2', 'HLA-DQB1', 'PPA1', 'TCTN3', 'APOL1', 'CAMK2D', 'HLA-DPA1', 'CD74', 'HLA-DQA1', 'CST3', 'BCL2L11', 'DHRS3', 'ID2', 'RAB11FIP1', 'QPRT', 'OSBPL9', 'PTPN22', 'PTPN7', 'FAM129A', 'DST', 'FUCA1', 'ANPEP', 'MARCKSL1', 'HLA-DRB1', 'CLIC2', 'STK17A', 'RGS1', 'HLA-DRA', 'BASP1', 'ACTN1', 'PDLIM7', 'CD40', 'AC093673.5', 'NCOA7', 'PDE4DIP', 'TUBA1C', 'STMN1', 'CSF2RA', 'CD38', 'MIR4435-1HG', 'DUSP2', 'LGMN', 'PKIB', 'VMO1', 'SPATS2L'],
    "cDC2_genes": ['FCER1A', 'PPP1R14A', 'CD1C', 'AKR1C3', 'ENHO', 'CD1E', 'SLC38A1', 'P2RY14', 'AFF3', 'FCGR2B', 'CLEC10A', 'NDRG2', 'CST7', 'PKIB', 'CACNA2D3', 'ADAM28', 'SDPR', 'PLD4', 'NET1', 'FLT3', 'GAS6', 'ALOX5AP', 'AGPAT9', 'HLA-DQA1', 'RGS1', 'FILIP1L', 'C12orf45', 'HLA-DQB1', 'IL1R2', 'ARHGAP5', 'HLA-DPB1', 'GPR183', 'AXL', 'ADAM8', 'USP53', 'SERPINF1', 'RGCC', 'HLA-DPA1', 'CD72', 'GSN', 'CLIC2', 'ACAP1', 'ATP1B1', 'HLA-DRA', 'PMAIP1', 'CD74', 'CSF2RA', 'AREG', 'PPA1', 'FPR3', 'HLA-DRB1', 'ALCAM', 'SPATS2L', 'ARL4C', 'PTMS', 'STMN1', 'LMNA', 'VSIG4'],
    "pDC_genes":['SCT', 'LRRC26', 'LILRA4', 'MAP1A', 'PTCRA', 'LAMP5', 'CLEC4C', 'MYBL2', 'SMIM5', 'KCNK17', 'PACSIN1', 'RP11-117D22.2', 'TCL1A', 'IL3RA', 'SPIB', 'PLD4', 'BCL11A', 'RASD1', 'FAM129C', 'TPM2', 'TSPAN13', 'PTPRS', 'SERPINF1', 'PPP1R14A', 'DERL3', 'SMPD3', 'IRF4', 'ITM2C', 'PFKFB2', 'PTGDS', 'FUT7', 'SOX4', 'IRF7', 'CXCR3', 'P2RY14', 'NOTCH4', 'PMEPA1', 'LINC00996', 'PPP1R14B', 'CLN8', 'VIPR2', 'AEBP1', 'C1orf186', 'MZB1', 'IRF8', 'TCF4', 'UGCG', 'GZMB', 'TNFRSF21', 'GAPT'],
    "MigcDC_genes": ['LAD1', 'CCR7', 'LAMP3', 'CCL19', 'CCL22', 'IL7R', 'SLC7A11', 'FSCN1', 'RAMP1', 'IDO1', 'CCL17', 'CD1B', 'MIR155HG', 'TNFRSF4', 'CXCL9', 'GPR157', 'DUSP4', 'IL4I1', 'EBI3', 'TRAF1', 'MARCKSL1', 'SPIB', 'ARHGAP10', 'HLA-DQB2', 'CD1E', 'MGLL', 'IL32', 'DUSP5', 'NRP2', 'CST7', 'PSD3', 'BIRC3', 'IRF4', 'C15orf48', 'HILPDA', 'GADD45A', 'SLC38A1', 'TRAF5', 'C12orf45', 'TXN', 'RAB9A', 'TBC1D4', 'TUBA1C', 'RGS1', 'SERPINF1', 'NMRK1', 'PPA1', 'CD40', 'GSN', 'POGLUT1', 'CLIC2', 'NAV1', 'PVRL2', 'PTGIR', 'RP11-138A9.2', 'FABP5', 'ALCAM', 'FAM129A', 'NET1', 'CSF2RA', 'ANTXR2', 'UGCG', 'GPR137B', 'CD1C', 'ADAM8', 'RELB', 'PALLD', 'SLAMF7', 'ST3GAL6', 'RP11-356I2.4', 'GPR183', 'HLA-DQA1', 'NFKB1', 'FLT3', 'RP11-138A9.1', 'CERS6', 'DNASE1L3', 'P2RY8', 'CD83', 'MAFF', 'SESN1', 'CYB5A', 'PTMS', 'SDC2', 'CCL5', 'ATP1B1', 'ETV3', 'ATF5', 'HLA-DQB1', 'FILIP1L', 'PMAIP1', 'PIM3', 'ID2', 'MARCKS', 'EPSTI1', 'BHLHE40', 'SOCS1', 'BASP1', 'HLA-DPB1', 'CD74', 'APOC1', 'HLA-DRA', 'CD72', 'GBP1', 'NR4A3', 'HLA-DPA1', 'PLEK', 'LMNA', 'CLEC10A'],
    "Monocyte_genes":['S100A8', 'S100A12', 'S100A9', 'VCAN', 'CTB-61M7.2', 'THBS1', 'EREG', 'MGST1', 'RP11-1143G9.4', 'ASGR1', 'BST1', 'AQP9', 'FCN1', 'CDA', 'LYZ', 'SLC2A3', 'MEGF9', 'APOBEC3A', 'CD300E', 'CD36', 'RAB27A', 'CLEC4E', 'TMEM176A', 'SELL', 'LGALS2', 'CXCL2', 'ITGAM', 'CSF3R', 'G0S2', 'EGR1', 'SOCS3', 'CCDC69', 'CAPG', 'TREM1', 'SLC11A1', 'AREG'],
    "Patmono_genes":['LYPD2', 'CDKN1C', 'CKB', 'PTP4A3', 'CD79B', 'HES4', 'PPM1N', 'SYTL1', 'FCGR3A', 'LILRA3', 'RHOC', 'RP11-290F20.3', 'TESC', 'SPN', 'TCF7L2', 'TPPP3', 'SLC2A6', 'SLC44A2', 'RRAS', 'SNX18', 'OAS1', 'FAM110A', 'VMO1', 'MTSS1', 'LILRB2', 'PHF19', 'TBC1D10C', 'FPR2', 'LILRA5', 'IFITM2', 'SIGLEC10', 'MYO1G', 'PAG1', 'SIDT2', 'LTB', 'LST1', 'S1PR4', 'ITGAL', 'STXBP2', 'CNIH4', 'GCH1', 'IFITM3', 'KLF2'],
    "LAM_genes":['FN1', 'CST6', 'SPP1', 'MT1G', 'TREM2', 'GPNMB', 'SLC16A10', 'CD9', 'EMP1', 'APOC1', 'LMNA', 'C15orf48', 'CLEC5A', 'IL7R', 'FABP5', 'RNASE1', 'FBP1', 'SDS', 'RAB13', 'OLR1', 'APOE', 'RGCC', 'GPX3', 'TSC22D1', 'SLC39A8', 'CAPG', 'ACP5', 'MMP19', 'ALCAM', 'GSN', 'FPR3', 'PDE4DIP', 'MT1X', 'LHFPL2', 'SDC2', 'MITF', 'PLA2G7', 'LGALS3', 'PHLDA1', 'ANKRD28', 'AVPI1', 'PTMS', 'MARCO', 'B3GNT5', 'PLTP', 'GPR183', 'DAB2', 'CD59', 'RGS1', 'LGMN'],
    "MATLAM_genes":['TREM2', 'GPNMB', 'CD9', 'RNASE1', 'OTOA', 'SDS', 'A2M', 'APOC1', 'LGMN', 'FCGBP', 'FABP5', 'SLCO2B1', 'APOE', 'C1QC', 'PLA2G7', 'FPR3', 'ID3', 'DAB2', 'AXL', 'C1QB', 'PLTP', 'RGS1', 'LTC4S', 'CD72', 'GATM', 'MGLL', 'LMNA', 'ABCC5', 'GAS6', 'ANKH', 'STMN1', 'C1QA', 'ALCAM', 'MSR1', 'PKIB', 'PHLDA1', 'ATP1B1', 'GPR34', 'PTMS', 'RGCC', 'MGAT4A', 'FCGR2B', 'LIPA', 'ACP5', 'CTSL', 'MITF', 'MAF', 'EBI3', 'KCNMA1', 'MERTK', 'VSIG4', 'C15orf48', 'FILIP1L', 'SEPP1', 'SLC40A1', 'CD81', 'GPR183', 'SDC2', 'GSN', 'FOLR2', 'CD59', 'CD84', 'CPM', 'MS4A4A', 'ABCA1', 'HLA-DQA1', 'CLEC10A', 'CLIC2', 'ARL4C', 'RGL1', 'EPB41L2', 'MMP19', 'AVPI1', 'ALB', 'C1orf54', 'PDK4', 'CD1C', 'VMO1'],
    "MAC1_genes":['IGSF21', 'FCGBP', 'USP53', 'ID3', 'C1QC', 'A2M', 'SDS', 'PLTP', 'MRC1L1', 'SLCO2B1', 'LTC4S', 'CD9', 'FCGR2B', 'RGS1', 'VSIG4', 'FPR3', 'C1QB', 'GPR34', 'AXL', 'FILIP1L', 'PKIB', 'C1QA', 'CD72', 'CLEC9A', 'NDRG2', 'CLEC10A', 'GATM', 'ALB', 'GPR183', 'MSR1', 'HLA-DPA1', 'PAPOLG', 'DAB2', 'PLD4', 'LPAR6', 'PHLDA1', 'EBI3', 'F13A1', 'FCER1A', 'HLA-DQA1', 'CPM', 'HLA-DPB1', 'ALOX5AP', 'SERPINF1', 'HLA-DRB1', 'RCAN1', 'STMN1', 'HLA-DRA', 'CCL3L1', 'ATP1B1', 'HLA-DQB1', 'PRDM1', 'MAF', 'ALCAM', 'C1orf54', 'APOC1', 'PTMS', 'LGALS3BP', 'CLIC2', 'RGCC', 'SLC40A1', 'VMO1', 'SERPING1', 'EPB41L2', 'CCL3', 'FOLR2', 'CD1C', 'PDK4', 'SEPP1', 'APOE'],
    "moKC_genes":['RNASE1', 'IGSF21', 'FCGBP', 'LILRB5', 'SLCO2B1', 'C1QC', 'SEPP1', 'PLTP', 'MRC1L1', 'NRP1', 'A2M', 'C1QB', 'TMEM37', 'LYVE1', 'FRMD4A', 'GFRA2', 'APOE', 'FOLR2', 'GPR34', 'OLFML2B', 'C1QA', 'LGMN', 'MAF', 'ID3', 'CXCL12', 'AXL', 'MERTK', 'SLC40A1', 'DAB2', 'MRC1', 'DHRS3', 'PDK4', 'GATM', 'LTC4S', 'MS4A4A', 'GPNMB', 'APOC1', 'USP53', 'EBI3', 'FPR3', 'FILIP1L', 'ALB', 'STAB1', 'SDC3', 'SIGLEC1', 'KCNMA1', 'VSIG4', 'ANKH', 'PKIB', 'MSR1', 'VCAM1', 'FCHO2', 'CCL3', 'SERPING1', 'CLEC9A', 'LGALS3BP', 'CCL4', 'CCL3L1', 'CD72', 'STMN1', 'CPM', 'CD163', 'RGL1', 'ABCA1', 'CD5L', 'EPS8', 'C2', 'CLIC2', 'RGS1', 'SPRED1', 'ATP1B1', 'TSPAN4', 'RCAN1', 'LIPA', 'SMPDL3A', 'WWP1', 'FRMD4B', 'EPB41L2', 'CD81', 'TCF4', 'AP2A2', 'SPATS2L', 'CD84', 'DST', 'FMNL2', 'MS4A7', 'CD59', 'CTSL', 'PLD3', 'HLA-DPA1', 'PTMS', 'ARL4C', 'VMO1', 'C1orf54', 'EGR2', 'CLEC10A', 'FCGR3A'],
    "KC_genes":['TIMD4', 'SLC16A9', 'NDST3', 'KCNAB1', 'ITGAD', 'CD5L', 'VCAM1', 'SELENBP1', 'BCAM', 'CDH5', 'CETP', 'RND3', 'FEZ1', 'FABP3', 'SDC3', 'CXCL12', 'ITLN1', 'CTD-2337J16.1', 'LYVE1', 'SEPP1', 'TMEM37', 'MARCO', 'SCD', 'LILRB5', 'PPAP2B', 'ITGA9', 'IGF1', 'APOE', 'SLC7A8', 'EPAS1', 'SLC40A1', 'FRMD4A', 'SUCNR1', 'ACVRL1', 'IGFBP4', 'ME1', 'LGMN', 'SIGLEC1', 'SIGLEC11', 'GPR126', 'FOLR2', 'LAG3', 'ITGB5', 'NRP1', 'C1QB', 'AXL', 'MRC1', 'C1QA', 'IFI27', 'C1QC', 'C2', 'SLC46A1', 'SLCO2B1', 'ETV5', 'SCN1B', 'NR1H3', 'DHRS3', 'PDK4', 'CD209', 'CCL4L1', 'GPNMB', 'ARHGEF12', 'CD59', 'RGL1', 'ANKRD36C', 'CCL4', 'MAF', 'EPB41L2', 'GATM', 'CCND1', 'CTSF', 'IFT74', 'LIPA', 'A2M', 'MYO9A', 'ATP1B1', 'CCL4L2', 'IFIT1', 'SERPING1', 'SPATS2L', 'MERTK', 'FUCA1', 'ADORA3', 'SMPDL3A', 'CCL3', 'ABCA1', 'MRC1L1', 'ANKH', 'CMKLR1', 'DST', 'PLD3', 'PLTP', 'FSCN1', 'APOC1', 'MMD', 'QPRT', 'VSIG4', 'RHOBTB3', 'EXT1', 'DAB2', 'ALB', 'RCAN1', 'FAM213A', 'EBI3', 'GFRA2', 'CD82', 'CCL3L3', 'PLA2G15', 'WWP1', 'CD81', 'ADM', 'ABCG1', 'FXYD6', 'STOM', 'KCNMA1', 'IFIT2', 'CD38', 'PPP1R12B', 'APOC3', 'CTSL', 'LGALS3BP', 'CR1', 'MSR1', 'TNFRSF21', 'CD163', 'GCNT1', 'EPHX1', 'SASH1', 'FILIP1L', 'SNX24', 'CLIC2', 'IFIT3', 'C1orf54', 'FPR3', 'FRMD4B', 'LTC4S', 'MMP19', 'CREG1', 'HMOX1', 'ZNF189', 'SDPR', 'NCEH1', 'FMNL2', 'CCL3L1', 'ITGAV', 'TSPAN4', 'ALDH1A1', 'NFIA', 'CTSB', 'PDE4DIP', 'ARL4C', 'ARRDC3', 'STMN1', 'RBP7', 'MS4A4A', 'EGR2', 'CD84', 'TCF4', 'VMO1', 'DSC2', 'GPR34', 'FCHO2', 'SLC9A9', 'PLA2G7', 'ADAP2', 'SCARB2', 'IL10', 'FCGRT', 'PHACTR2', 'AP2A2', 'ELL2', 'ACP2', 'MS4A7', 'P4HA1', 'PDGFC', 'GPR137B', 'SGPP1', 'SPRED1', 'PKIB', 'IFI44L', 'PRNP', 'CD72', 'SIAH2', 'MS4A6A', 'EPS8', 'UGCG', 'IDH1', 'BLVRB', 'ICAM1', 'RGS1', 'CTSD', 'CD63', 'FCGR3A', 'MT1F', 'GIMAP5', 'KCNJ2']
    }
sc.tl.score_genes(adata_snmono, gene_list=gene_sets["cDC1_genes"] , score_name="cDC1_score")
sc.pl.umap(adata_snmono, color='cDC1_score', cmap='bwr', title='cDC1_score')
sc.tl.score_genes(adata_snmono, gene_list=gene_sets["cDC2_genes"] , score_name="cDC2_score")
sc.pl.umap(adata_snmono, color='cDC2_score', cmap='bwr', title='cDC2_score')
sc.tl.score_genes(adata_snmono, gene_list=gene_sets["MigcDC_genes"] , score_name="MigcDC_score")
sc.pl.umap(adata_snmono, color='MigcDC_score', cmap='bwr', title='MigcDC_score')
sc.tl.score_genes(adata_snmono, gene_list=gene_sets["Monocyte_genes"] , score_name="Monocyte_score")
sc.pl.umap(adata_snmono, color='Monocyte_score', cmap='bwr', title='Monocyte_score')
sc.tl.score_genes(adata_snmono, gene_list=gene_sets["Patmono_genes"] , score_name="Patmono_score")
sc.pl.umap(adata_snmono, color='Patmono_score', cmap='bwr', title='Patmono_score')
sc.tl.score_genes(adata_snmono, gene_list=gene_sets["LAM_genes"] , score_name="LAM_score")
sc.pl.umap(adata_snmono, color='LAM_score', cmap='bwr', title='LAM_score')
sc.tl.score_genes(adata_snmono, gene_list=gene_sets["MATLAM_genes"] , score_name="MATLAM_score")
sc.pl.umap(adata_snmono, color='MATLAM_score', cmap='bwr', title='MATLAM_score')
sc.tl.score_genes(adata_snmono, gene_list=gene_sets["MAC1_genes"] , score_name="MAC1_score")
sc.pl.umap(adata_snmono, color='MAC1_score', cmap='bwr', title='MAC1_score')
sc.tl.score_genes(adata_snmono, gene_list=gene_sets["moKC_genes"] , score_name="moKC_score")
sc.pl.umap(adata_snmono, color='moKC_score', cmap='bwr', title='moKC_score')
sc.tl.score_genes(adata_snmono, gene_list=gene_sets["KC_genes"] , score_name="KC_score")
sc.pl.umap(adata_snmono, color='KC_score', cmap='bwr', title='KC_score')

In [ ]:
sc.tl.leiden(adata_snmono, resolution=0.5, key_added="10_cluster")
sc.pl.umap(adata_snmono, color=['10_cluster'])

In [ ]:
adata_snmono

In [ ]:
predictions_cluster = celltypist.annotate(adata_snmono, model='Healthy_Human_Liver.pkl', majority_voting=True, over_clustering=adata_snmono.obs["10_cluster"]
)
adata_snmono =predictions_cluster.to_adata()
adata_snmono.obs['celltypist_liver'] = adata_snmono.obs['majority_voting']
sc.pl.umap(
    adata_snmono,
    color='celltypist_liver',
    #legend_loc='on data',
    legend_fontsize=10,
    frameon=False,
    title='state',
    size=15
)

In [ ]:
predictions_cluster = celltypist.annotate(adata_snmono, model='Immune_All_Low.pkl', majority_voting=True, over_clustering=adata_snmono.obs["10_cluster"]
)
adata_snmono =predictions_cluster.to_adata()
adata_snmono.obs['Immune_low'] = adata_snmono.obs['majority_voting']
sc.pl.umap(
    adata_snmono,
    color='Immune_low',
    #legend_loc='on data',
    legend_fontsize=10,
    frameon=False,
    title='state',
    size=15
)

In [ ]:
predictions_cluster = celltypist.annotate(adata_snmono, model='Immune_All_High.pkl', majority_voting=True, over_clustering=adata_snmono.obs["10_cluster"]
)
adata_snmono =predictions_cluster.to_adata()
adata_snmono.obs['Immune_high'] = adata_snmono.obs['majority_voting']
sc.pl.umap(
    adata_snmono,
    color='Immune_high',
    #legend_loc='on data',
    legend_fontsize=10,
    frameon=False,
    title='state',
    size=15
)

In [ ]:
sc.tl.dendrogram(adata_snmono,groupby='10_cluster')
sc.tl.rank_genes_groups(adata_snmono, groupby='10_cluster', method='wilcoxon')
sc.pl.rank_genes_groups_dotplot(adata_snmono, n_genes=10)

In [ ]:
rank_df = sc.get.rank_genes_groups_df(adata_snmono, group=None)
top_genes = rank_df.groupby("group")["names"].apply(lambda x: x.head(20)).explode().unique().tolist()
sc.pl.heatmap(
    adata_snmono,
    var_names=top_genes,
    groupby="10_cluster",
    use_raw=False,
    cmap="bwr",
    vmin=-3,
    vmax=3,
    swap_axes=True,
    dendrogram=False,
    show=True,
    figsize=(10, 7)
)

In [ ]:
# 그룹별 샘플링 수
n_cells_per_group = 50
groups = adata_snmono.obs['10_cluster'].unique()

# 샘플링된 셀들의 인덱스 저장
selected_indices = []

for group in groups:
    group_cells = adata_snmono.obs[adata_snmono.obs['10_cluster'] == group].index
    n = min(len(group_cells), n_cells_per_group)  # 그룹 크기가 작을 경우 자동 조절
    sampled = np.random.choice(group_cells, n, replace=False)
    selected_indices.extend(sampled)

adata_sampled = adata_snmono[selected_indices]
fig_dict = sc.pl.rank_genes_groups_heatmap(
    adata_sampled,
    n_genes=10,
    groupby="10_cluster",
    use_raw=False,
    swap_axes=True,
    vmin=-3,
    vmax=3,
    dendrogram=False,
    figsize=(10, 7),
    cmap="bwr",
    show=False,
    show_gene_labels=False,
    return_fig=True
)

print(type(fig_dict))
print(fig_dict.keys())

In [ ]:
adata_snmono.obs['state']

In [ ]:
from scipy.stats import fisher_exact
import pandas as pd

# 상태 및 클러스터 정보
state = adata_snmono.obs['state']
cluster = adata_snmono.obs['10_cluster']

# 클러스터(세포타입) 목록
clusters = cluster.unique()
results = []

# Donor / IPF 구분
for clus in sorted(clusters):  # 문자형 label일 경우 그냥 sorted() 사용
    # 각 조건별 개수 계산
    donor_in = ((state == "normal") & (cluster == clus)).sum()
    donor_out = ((state == "normal") & (cluster != clus)).sum()
    ipf_in = ((state == "MASLD") & (cluster == clus)).sum()
    ipf_out = ((state == "MASLD") & (cluster != clus)).sum()
    
    table = [[donor_in, donor_out], [ipf_in, ipf_out]]
    oddsratio, pval = fisher_exact(table)
    
    # 비율 계산
    donor_frac = donor_in / (donor_in + donor_out) if (donor_in + donor_out) > 0 else 0
    ipf_frac = ipf_in / (ipf_in + ipf_out) if (ipf_in + ipf_out) > 0 else 0
    
    results.append({
        "cluster": clus,
        "donor_fraction": donor_frac,
        "ipf_fraction": ipf_frac,
        "odds_ratio": oddsratio,
        "p_value": pval
    })

# 결과를 DataFrame으로 정리
df_result = pd.DataFrame(results)

# p-value 기준 정렬 (선택사항)
df_result = df_result.sort_values("cluster")
print(df_result)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plt.figure(figsize=(15, 6))
x = np.arange(len(df_result))
width = 0.35

# Bar plot
plt.bar(x - width/2, df_result['donor_fraction'], width, label='normal')
plt.bar(x + width/2, df_result['ipf_fraction'], width, label='MASLD')

# X축 설정
plt.xticks(x, df_result['cluster'])
plt.ylabel('Fraction')
plt.title('T cell cluster proportion by condition')
plt.legend()

# p-value 별로 * 표시
for i, pval in enumerate(df_result['p_value']):
    if pval < 0.001:
        stars = '***'
    elif pval < 0.01:
        stars = '**'
    elif pval < 0.05:
        stars = '*'
    else:
        stars = ''
    
    if stars:
        max_height = max(df_result['donor_fraction'][i], df_result['ipf_fraction'][i])
        plt.text(x[i], 0.55, stars, ha='center', va='bottom', fontsize=12, color='black')

plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 2))
sc.pl.violin(adata_snmono, keys = 'SLC7A5',groupby='10_cluster',ax=ax)
plt.show()

In [ ]:
# 2. crosstab 생성
cluster_labels = adata_snmono.obs['10_cluster']
batch_labels = adata_snmono.obs['state']
crosstab = pd.crosstab(batch_labels, cluster_labels, normalize='index')

# 3. matplotlib barplot 그릴 때 palette 지정
crosstab.plot(
    kind='bar',
    stacked=True,
    figsize=(5, 10),
    #color=[palette_dict[col] for col in crosstab.columns]
)

plt.title('Major cell lineages in MASLD vs Normal')
plt.ylabel('Proportion')
plt.xlabel('state')
plt.legend(title='MASLD vs Normal', bbox_to_anchor=(1.05, 1), loc='upper left')
#plt.savefig("/home/jaehyunchoi/bystander_collaboration/IPF_data/analysis/figures/3publicatlas_celltypist_IPF_state_stackedbargraph.pdf", bbox_inches='tight', dpi=300)

In [ ]:
adata_snmono=sc.read("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/snRNA_v1/snRNA_adata_mono02_celltypistsub.h5ad")

In [ ]:
adata_snmono.write("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/snRNA_v1/snRNA_adata_mono02_celltypistsub.h5ad")

In [ ]:
adata_snmono

In [ ]:
sc.pl.umap(adata_snmono, color = '10_cluster')

In [ ]:
Rama_genes ={'Hepatocyte': ['GHR', 'CYP2A7', 'PCK1', 'CYP2A6', 'HSD11B1', 'G6PC', 'BCHE', 'HAMP', 'HPR', 'SEC16B', 'MASP2', 'ACSS2', 'AKR1C1', 'ALDH6A1', 'HMGCS1'], 'Epithelia': ['ALB', 'AGXT', 'TF', 'TTR', 'KRT19', 'SOX9', 'KNG1', 'HSD17B6', 'F2', 'ITIH1', 'CYP4A11'], 'Mesenchyme': ['TIMP1', 'COL3A1', 'ACTA2'], 'Endothelia': ['EGFL7', 'HSPG2', 'OIT3', 'CLEC4G', 'EMCN', 'FCN3', 'CLEC4M', 'CLEC14A'], 'Tcell': ['CD2'], 'ILC': ['FGFBP2', 'PRF1', 'KLRF1', 'KLRC1'], 'B cell': ['CD79A', 'MS4A1'], 'pDC': ['LILRA4', 'PTCRA', 'LRRC26', 'CLEC4C'], 'Plasma cell': ['FCRL5', 'JSRP1'], 'MP': ['FCN1', 'IL1B', 'MS4A7', 'VCAN', 'CPVL', 'MNDA', 'CD163', 'CYBB', 'CSF2RA', 'MSR1', 'CLEC10A', 'C1QC']}
sc.pl.dotplot(
    adata_snmono,
    Rama_genes,
    groupby="10_cluster",
    standard_scale="var",
    figsize=(11, 8),
    #show=False,
    #return_fig=True
)

# 8 subclustering analysis

In [ ]:
adata_snraw=sc.read("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/snRNA_v1/snRNA_raw.h5ad")

In [ ]:
adata_snrna

In [ ]:
adata_snraw.obs['cluster'] = adata_snrna.obs['cluster']

In [ ]:
adata_snraw.obs['celltypist_liver'] = adata_snrna.obs['celltypist_liver']

In [ ]:
adata_snraw.obs['cluster'].value_counts()

In [ ]:
adata_snmono = adata_snraw[adata_snraw.obs['cluster'].isin(['8'])].copy()
adata_snmono

In [ ]:
adata_snmono.write("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/snRNA_v1/snRNA_adata_mono01.h5ad")

In [ ]:
sc.pp.normalize_total(adata_snmono, target_sum=1e4)
sc.pp.log1p(adata_snmono)
adata_snmono.raw = adata_snmono
sc.pp.highly_variable_genes(adata_snmono, flavor='seurat', n_top_genes=2000)

In [ ]:
# Scale data
sc.pp.scale(adata_snmono, max_value=10)
# Run PCA
sc.tl.pca(adata_snmono, svd_solver='arpack')
print(adata_snmono.obsm['X_pca'][:5, :5])
sc.pl.pca(adata_snmono, color='sample')

import matplotlib.pyplot as plt
var_ratio = adata_snmono.uns['pca']['variance_ratio']
plt.plot(range(1, len(var_ratio) + 1), np.cumsum(var_ratio), marker='o')
plt.xlabel('Number of Principal Components')
plt.ylabel('Cumulative Explained Variance')
plt.title('PCA Cumulative Variance')
plt.grid(True)
plt.show()

In [ ]:
# t-sne 
sc.tl.tsne(adata_snmono, use_rep='X_pca', n_pcs=30)
# umap
sc.pp.neighbors(adata_snmono, n_pcs=30, use_rep='X_pca')
sc.tl.umap(adata_snmono)
sc.pl.umap(adata_snmono, color=['sample'])

In [ ]:
# Harmony로 batch effect 제거
harmony_out = hm.run_harmony(adata_snmono.obsm['X_pca'],adata_snmono.obs, 'sample')

# Harmony 결과를 저장
adata_snmono.obsm['X_pca_harmony'] = harmony_out.Z_corr.T

# UMAP을 사용한 시각화
sc.pp.neighbors(adata_snmono, use_rep='X_pca_harmony')  # Harmony 결과를 사용
sc.tl.umap(adata_snmono)

In [ ]:
sc.pl.umap(adata_snmono,color='state')

In [ ]:
sc.tl.rank_genes_groups(
    adata_snmono,
    groupby="state",        # 비교할 obs 열
    groups=["MASLD"],     # 비교할 그룹
    reference="normal",     # 참조 그룹
    use_raw=True,
    method="wilcoxon"       # 또는 't-test', 'logreg'
)

In [ ]:
deg_df = sc.get.rank_genes_groups_df(adata_snmono, group="MASLD")
print(deg_df.head())

In [ ]:
filtered = deg_df[
    (deg_df['logfoldchanges'] > 0.58) &
    (deg_df['pvals_adj'] < 0.05)
]
top20_filtered = filtered.sort_values(by='scores', ascending=False).head(50)
print(top20_filtered)

In [ ]:
deg_df['-log10(padj)'] = -np.log10(deg_df['pvals_adj'])
genes_of_interest = ["SLC7A5"]

plt.figure(figsize=(7, 6))
plt.scatter(
    deg_df['logfoldchanges'], 
    deg_df['-log10(padj)'], 
    c="grey", s=10, alpha=0.7
)

# 유의미한 cutoff 설정 (예: logFC > 1, padj < 0.05)
sig_up = (deg_df['logfoldchanges'] > 1) & (deg_df['pvals_adj'] < 0.05)
sig_down = (deg_df['logfoldchanges'] < -1) & (deg_df['pvals_adj'] < 0.05)

plt.scatter(deg_df.loc[sig_up, 'logfoldchanges'],
            deg_df.loc[sig_up, '-log10(padj)'],
            c="red", s=10, label="Upregulated")

plt.scatter(deg_df.loc[sig_down, 'logfoldchanges'],
            deg_df.loc[sig_down, '-log10(padj)'],
            c="blue", s=10, label="Downregulated")

# cutoff 선
plt.axvline(1, color="black", linestyle="--")
plt.axvline(-1, color="black", linestyle="--")
plt.axhline(-np.log10(0.05), color="black", linestyle="--")

plt.xlabel("log2 Fold Change")
plt.ylabel("-log10 Adjusted p-value")
plt.title("Volcano Plot: MASLD vs Normal")
plt.legend()

for gene in genes_of_interest:
    if gene in deg_df['names'].values:
        x = deg_df.loc[deg_df['names'] == gene, 'logfoldchanges'].values[0]
        y = deg_df.loc[deg_df['names'] == gene, '-log10(padj)'].values[0]
        plt.text(x, y, gene, fontsize=8, color="black")  # 텍스트 표시


plt.tight_layout()
plt.show()

In [ ]:
sc.pl.umap(
    adata_snmono,
    color='state',
    #legend_loc='on data',
    legend_fontsize=10,
    frameon=False,
    title='state',
    size=15
)

In [ ]:
adata_snmono.obs_names_make_unique()

In [ ]:
sc.pl.umap(
    adata_snmono,
    color='SLC7A5',
    #legend_loc='on data',
    legend_fontsize=10,
    frameon=False,
    title='SLC7A5',
    size=15
)

In [ ]:
adata_snmono.write("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/snRNA_v1/snRNA_adata_mono02.h5ad")

In [ ]:
gene_sets = {
    "cDC1_genes": ['XCR1', 'CLNK', 'TACSTD2', 'CLEC9A', 'EGLN3', 'IDO1', 'DNASE1L3', 'CADM1', 'S100B', 'KIAA0226L', 'WDFY4', 'CCND1', 'DUSP4', 'P2RY14', 'C1orf54', 'NRARP', 'CPNE3', 'SLAMF7', 'GPR157', 'B3GNT7', 'FLT3', 'NET1', 'RGCC', 'SERPINF1', 'BATF3', 'NDRG2', 'SNX3', 'ADAM28', 'AIM2', 'IL18R1', 'GPR171', 'PTPRCAP', 'IRF8', 'CPVL', 'C12orf45', 'TPD52', 'SLC38A1', 'LMNA', 'CD59', 'C15orf48', 'NFATC2', 'NAV1', 'MAP2K6', 'PTMS', 'ARHGAP5', 'HLA-DPB1', 'CST7', 'LGALS2', 'HLA-DQB1', 'PPA1', 'TCTN3', 'APOL1', 'CAMK2D', 'HLA-DPA1', 'CD74', 'HLA-DQA1', 'CST3', 'BCL2L11', 'DHRS3', 'ID2', 'RAB11FIP1', 'QPRT', 'OSBPL9', 'PTPN22', 'PTPN7', 'FAM129A', 'DST', 'FUCA1', 'ANPEP', 'MARCKSL1', 'HLA-DRB1', 'CLIC2', 'STK17A', 'RGS1', 'HLA-DRA', 'BASP1', 'ACTN1', 'PDLIM7', 'CD40', 'AC093673.5', 'NCOA7', 'PDE4DIP', 'TUBA1C', 'STMN1', 'CSF2RA', 'CD38', 'MIR4435-1HG', 'DUSP2', 'LGMN', 'PKIB', 'VMO1', 'SPATS2L'],
    "cDC2_genes": ['FCER1A', 'PPP1R14A', 'CD1C', 'AKR1C3', 'ENHO', 'CD1E', 'SLC38A1', 'P2RY14', 'AFF3', 'FCGR2B', 'CLEC10A', 'NDRG2', 'CST7', 'PKIB', 'CACNA2D3', 'ADAM28', 'SDPR', 'PLD4', 'NET1', 'FLT3', 'GAS6', 'ALOX5AP', 'AGPAT9', 'HLA-DQA1', 'RGS1', 'FILIP1L', 'C12orf45', 'HLA-DQB1', 'IL1R2', 'ARHGAP5', 'HLA-DPB1', 'GPR183', 'AXL', 'ADAM8', 'USP53', 'SERPINF1', 'RGCC', 'HLA-DPA1', 'CD72', 'GSN', 'CLIC2', 'ACAP1', 'ATP1B1', 'HLA-DRA', 'PMAIP1', 'CD74', 'CSF2RA', 'AREG', 'PPA1', 'FPR3', 'HLA-DRB1', 'ALCAM', 'SPATS2L', 'ARL4C', 'PTMS', 'STMN1', 'LMNA', 'VSIG4'],
    "pDC_genes":['SCT', 'LRRC26', 'LILRA4', 'MAP1A', 'PTCRA', 'LAMP5', 'CLEC4C', 'MYBL2', 'SMIM5', 'KCNK17', 'PACSIN1', 'RP11-117D22.2', 'TCL1A', 'IL3RA', 'SPIB', 'PLD4', 'BCL11A', 'RASD1', 'FAM129C', 'TPM2', 'TSPAN13', 'PTPRS', 'SERPINF1', 'PPP1R14A', 'DERL3', 'SMPD3', 'IRF4', 'ITM2C', 'PFKFB2', 'PTGDS', 'FUT7', 'SOX4', 'IRF7', 'CXCR3', 'P2RY14', 'NOTCH4', 'PMEPA1', 'LINC00996', 'PPP1R14B', 'CLN8', 'VIPR2', 'AEBP1', 'C1orf186', 'MZB1', 'IRF8', 'TCF4', 'UGCG', 'GZMB', 'TNFRSF21', 'GAPT'],
    "MigcDC_genes": ['LAD1', 'CCR7', 'LAMP3', 'CCL19', 'CCL22', 'IL7R', 'SLC7A11', 'FSCN1', 'RAMP1', 'IDO1', 'CCL17', 'CD1B', 'MIR155HG', 'TNFRSF4', 'CXCL9', 'GPR157', 'DUSP4', 'IL4I1', 'EBI3', 'TRAF1', 'MARCKSL1', 'SPIB', 'ARHGAP10', 'HLA-DQB2', 'CD1E', 'MGLL', 'IL32', 'DUSP5', 'NRP2', 'CST7', 'PSD3', 'BIRC3', 'IRF4', 'C15orf48', 'HILPDA', 'GADD45A', 'SLC38A1', 'TRAF5', 'C12orf45', 'TXN', 'RAB9A', 'TBC1D4', 'TUBA1C', 'RGS1', 'SERPINF1', 'NMRK1', 'PPA1', 'CD40', 'GSN', 'POGLUT1', 'CLIC2', 'NAV1', 'PVRL2', 'PTGIR', 'RP11-138A9.2', 'FABP5', 'ALCAM', 'FAM129A', 'NET1', 'CSF2RA', 'ANTXR2', 'UGCG', 'GPR137B', 'CD1C', 'ADAM8', 'RELB', 'PALLD', 'SLAMF7', 'ST3GAL6', 'RP11-356I2.4', 'GPR183', 'HLA-DQA1', 'NFKB1', 'FLT3', 'RP11-138A9.1', 'CERS6', 'DNASE1L3', 'P2RY8', 'CD83', 'MAFF', 'SESN1', 'CYB5A', 'PTMS', 'SDC2', 'CCL5', 'ATP1B1', 'ETV3', 'ATF5', 'HLA-DQB1', 'FILIP1L', 'PMAIP1', 'PIM3', 'ID2', 'MARCKS', 'EPSTI1', 'BHLHE40', 'SOCS1', 'BASP1', 'HLA-DPB1', 'CD74', 'APOC1', 'HLA-DRA', 'CD72', 'GBP1', 'NR4A3', 'HLA-DPA1', 'PLEK', 'LMNA', 'CLEC10A'],
    "Monocyte_genes":['S100A8', 'S100A12', 'S100A9', 'VCAN', 'CTB-61M7.2', 'THBS1', 'EREG', 'MGST1', 'RP11-1143G9.4', 'ASGR1', 'BST1', 'AQP9', 'FCN1', 'CDA', 'LYZ', 'SLC2A3', 'MEGF9', 'APOBEC3A', 'CD300E', 'CD36', 'RAB27A', 'CLEC4E', 'TMEM176A', 'SELL', 'LGALS2', 'CXCL2', 'ITGAM', 'CSF3R', 'G0S2', 'EGR1', 'SOCS3', 'CCDC69', 'CAPG', 'TREM1', 'SLC11A1', 'AREG'],
    "Patmono_genes":['LYPD2', 'CDKN1C', 'CKB', 'PTP4A3', 'CD79B', 'HES4', 'PPM1N', 'SYTL1', 'FCGR3A', 'LILRA3', 'RHOC', 'RP11-290F20.3', 'TESC', 'SPN', 'TCF7L2', 'TPPP3', 'SLC2A6', 'SLC44A2', 'RRAS', 'SNX18', 'OAS1', 'FAM110A', 'VMO1', 'MTSS1', 'LILRB2', 'PHF19', 'TBC1D10C', 'FPR2', 'LILRA5', 'IFITM2', 'SIGLEC10', 'MYO1G', 'PAG1', 'SIDT2', 'LTB', 'LST1', 'S1PR4', 'ITGAL', 'STXBP2', 'CNIH4', 'GCH1', 'IFITM3', 'KLF2'],
    "LAM_genes":['FN1', 'CST6', 'SPP1', 'MT1G', 'TREM2', 'GPNMB', 'SLC16A10', 'CD9', 'EMP1', 'APOC1', 'LMNA', 'C15orf48', 'CLEC5A', 'IL7R', 'FABP5', 'RNASE1', 'FBP1', 'SDS', 'RAB13', 'OLR1', 'APOE', 'RGCC', 'GPX3', 'TSC22D1', 'SLC39A8', 'CAPG', 'ACP5', 'MMP19', 'ALCAM', 'GSN', 'FPR3', 'PDE4DIP', 'MT1X', 'LHFPL2', 'SDC2', 'MITF', 'PLA2G7', 'LGALS3', 'PHLDA1', 'ANKRD28', 'AVPI1', 'PTMS', 'MARCO', 'B3GNT5', 'PLTP', 'GPR183', 'DAB2', 'CD59', 'RGS1', 'LGMN'],
    "MATLAM_genes":['TREM2', 'GPNMB', 'CD9', 'RNASE1', 'OTOA', 'SDS', 'A2M', 'APOC1', 'LGMN', 'FCGBP', 'FABP5', 'SLCO2B1', 'APOE', 'C1QC', 'PLA2G7', 'FPR3', 'ID3', 'DAB2', 'AXL', 'C1QB', 'PLTP', 'RGS1', 'LTC4S', 'CD72', 'GATM', 'MGLL', 'LMNA', 'ABCC5', 'GAS6', 'ANKH', 'STMN1', 'C1QA', 'ALCAM', 'MSR1', 'PKIB', 'PHLDA1', 'ATP1B1', 'GPR34', 'PTMS', 'RGCC', 'MGAT4A', 'FCGR2B', 'LIPA', 'ACP5', 'CTSL', 'MITF', 'MAF', 'EBI3', 'KCNMA1', 'MERTK', 'VSIG4', 'C15orf48', 'FILIP1L', 'SEPP1', 'SLC40A1', 'CD81', 'GPR183', 'SDC2', 'GSN', 'FOLR2', 'CD59', 'CD84', 'CPM', 'MS4A4A', 'ABCA1', 'HLA-DQA1', 'CLEC10A', 'CLIC2', 'ARL4C', 'RGL1', 'EPB41L2', 'MMP19', 'AVPI1', 'ALB', 'C1orf54', 'PDK4', 'CD1C', 'VMO1'],
    "MAC1_genes":['IGSF21', 'FCGBP', 'USP53', 'ID3', 'C1QC', 'A2M', 'SDS', 'PLTP', 'MRC1L1', 'SLCO2B1', 'LTC4S', 'CD9', 'FCGR2B', 'RGS1', 'VSIG4', 'FPR3', 'C1QB', 'GPR34', 'AXL', 'FILIP1L', 'PKIB', 'C1QA', 'CD72', 'CLEC9A', 'NDRG2', 'CLEC10A', 'GATM', 'ALB', 'GPR183', 'MSR1', 'HLA-DPA1', 'PAPOLG', 'DAB2', 'PLD4', 'LPAR6', 'PHLDA1', 'EBI3', 'F13A1', 'FCER1A', 'HLA-DQA1', 'CPM', 'HLA-DPB1', 'ALOX5AP', 'SERPINF1', 'HLA-DRB1', 'RCAN1', 'STMN1', 'HLA-DRA', 'CCL3L1', 'ATP1B1', 'HLA-DQB1', 'PRDM1', 'MAF', 'ALCAM', 'C1orf54', 'APOC1', 'PTMS', 'LGALS3BP', 'CLIC2', 'RGCC', 'SLC40A1', 'VMO1', 'SERPING1', 'EPB41L2', 'CCL3', 'FOLR2', 'CD1C', 'PDK4', 'SEPP1', 'APOE'],
    "moKC_genes":['RNASE1', 'IGSF21', 'FCGBP', 'LILRB5', 'SLCO2B1', 'C1QC', 'SEPP1', 'PLTP', 'MRC1L1', 'NRP1', 'A2M', 'C1QB', 'TMEM37', 'LYVE1', 'FRMD4A', 'GFRA2', 'APOE', 'FOLR2', 'GPR34', 'OLFML2B', 'C1QA', 'LGMN', 'MAF', 'ID3', 'CXCL12', 'AXL', 'MERTK', 'SLC40A1', 'DAB2', 'MRC1', 'DHRS3', 'PDK4', 'GATM', 'LTC4S', 'MS4A4A', 'GPNMB', 'APOC1', 'USP53', 'EBI3', 'FPR3', 'FILIP1L', 'ALB', 'STAB1', 'SDC3', 'SIGLEC1', 'KCNMA1', 'VSIG4', 'ANKH', 'PKIB', 'MSR1', 'VCAM1', 'FCHO2', 'CCL3', 'SERPING1', 'CLEC9A', 'LGALS3BP', 'CCL4', 'CCL3L1', 'CD72', 'STMN1', 'CPM', 'CD163', 'RGL1', 'ABCA1', 'CD5L', 'EPS8', 'C2', 'CLIC2', 'RGS1', 'SPRED1', 'ATP1B1', 'TSPAN4', 'RCAN1', 'LIPA', 'SMPDL3A', 'WWP1', 'FRMD4B', 'EPB41L2', 'CD81', 'TCF4', 'AP2A2', 'SPATS2L', 'CD84', 'DST', 'FMNL2', 'MS4A7', 'CD59', 'CTSL', 'PLD3', 'HLA-DPA1', 'PTMS', 'ARL4C', 'VMO1', 'C1orf54', 'EGR2', 'CLEC10A', 'FCGR3A'],
    "KC_genes":['TIMD4', 'SLC16A9', 'NDST3', 'KCNAB1', 'ITGAD', 'CD5L', 'VCAM1', 'SELENBP1', 'BCAM', 'CDH5', 'CETP', 'RND3', 'FEZ1', 'FABP3', 'SDC3', 'CXCL12', 'ITLN1', 'CTD-2337J16.1', 'LYVE1', 'SEPP1', 'TMEM37', 'MARCO', 'SCD', 'LILRB5', 'PPAP2B', 'ITGA9', 'IGF1', 'APOE', 'SLC7A8', 'EPAS1', 'SLC40A1', 'FRMD4A', 'SUCNR1', 'ACVRL1', 'IGFBP4', 'ME1', 'LGMN', 'SIGLEC1', 'SIGLEC11', 'GPR126', 'FOLR2', 'LAG3', 'ITGB5', 'NRP1', 'C1QB', 'AXL', 'MRC1', 'C1QA', 'IFI27', 'C1QC', 'C2', 'SLC46A1', 'SLCO2B1', 'ETV5', 'SCN1B', 'NR1H3', 'DHRS3', 'PDK4', 'CD209', 'CCL4L1', 'GPNMB', 'ARHGEF12', 'CD59', 'RGL1', 'ANKRD36C', 'CCL4', 'MAF', 'EPB41L2', 'GATM', 'CCND1', 'CTSF', 'IFT74', 'LIPA', 'A2M', 'MYO9A', 'ATP1B1', 'CCL4L2', 'IFIT1', 'SERPING1', 'SPATS2L', 'MERTK', 'FUCA1', 'ADORA3', 'SMPDL3A', 'CCL3', 'ABCA1', 'MRC1L1', 'ANKH', 'CMKLR1', 'DST', 'PLD3', 'PLTP', 'FSCN1', 'APOC1', 'MMD', 'QPRT', 'VSIG4', 'RHOBTB3', 'EXT1', 'DAB2', 'ALB', 'RCAN1', 'FAM213A', 'EBI3', 'GFRA2', 'CD82', 'CCL3L3', 'PLA2G15', 'WWP1', 'CD81', 'ADM', 'ABCG1', 'FXYD6', 'STOM', 'KCNMA1', 'IFIT2', 'CD38', 'PPP1R12B', 'APOC3', 'CTSL', 'LGALS3BP', 'CR1', 'MSR1', 'TNFRSF21', 'CD163', 'GCNT1', 'EPHX1', 'SASH1', 'FILIP1L', 'SNX24', 'CLIC2', 'IFIT3', 'C1orf54', 'FPR3', 'FRMD4B', 'LTC4S', 'MMP19', 'CREG1', 'HMOX1', 'ZNF189', 'SDPR', 'NCEH1', 'FMNL2', 'CCL3L1', 'ITGAV', 'TSPAN4', 'ALDH1A1', 'NFIA', 'CTSB', 'PDE4DIP', 'ARL4C', 'ARRDC3', 'STMN1', 'RBP7', 'MS4A4A', 'EGR2', 'CD84', 'TCF4', 'VMO1', 'DSC2', 'GPR34', 'FCHO2', 'SLC9A9', 'PLA2G7', 'ADAP2', 'SCARB2', 'IL10', 'FCGRT', 'PHACTR2', 'AP2A2', 'ELL2', 'ACP2', 'MS4A7', 'P4HA1', 'PDGFC', 'GPR137B', 'SGPP1', 'SPRED1', 'PKIB', 'IFI44L', 'PRNP', 'CD72', 'SIAH2', 'MS4A6A', 'EPS8', 'UGCG', 'IDH1', 'BLVRB', 'ICAM1', 'RGS1', 'CTSD', 'CD63', 'FCGR3A', 'MT1F', 'GIMAP5', 'KCNJ2']
    }


In [ ]:
sc.tl.score_genes(adata_snmono, gene_list=gene_sets["cDC1_genes"] , score_name="cDC1_score")
sc.pl.umap(adata_snmono, color='cDC1_score', cmap='bwr', title='cDC1_score')

In [ ]:
sc.tl.score_genes(adata_snmono, gene_list=gene_sets["cDC2_genes"] , score_name="cDC2_score")
sc.pl.umap(adata_snmono, color='cDC2_score', cmap='bwr', title='cDC2_score')

In [ ]:
sc.tl.score_genes(adata_snmono, gene_list=gene_sets["MigcDC_genes"] , score_name="MigcDC_score")
sc.pl.umap(adata_snmono, color='MigcDC_score', cmap='bwr', title='MigcDC_score')

In [ ]:
sc.tl.score_genes(adata_snmono, gene_list=gene_sets["Monocyte_genes"] , score_name="Monocyte_score")
sc.pl.umap(adata_snmono, color='Monocyte_score', cmap='bwr', title='Monocyte_score')

In [ ]:
sc.tl.score_genes(adata_snmono, gene_list=gene_sets["Patmono_genes"] , score_name="Patmono_score")
sc.pl.umap(adata_snmono, color='Patmono_score', cmap='bwr', title='Patmono_score')

In [ ]:
sc.tl.score_genes(adata_snmono, gene_list=gene_sets["LAM_genes"] , score_name="LAM_score")
sc.pl.umap(adata_snmono, color='LAM_score', cmap='bwr', title='LAM_score')

In [ ]:
sc.tl.score_genes(adata_snmono, gene_list=gene_sets["MATLAM_genes"] , score_name="MATLAM_score")
sc.pl.umap(adata_snmono, color='MATLAM_score', cmap='bwr', title='MATLAM_score')

In [ ]:
sc.tl.score_genes(adata_snmono, gene_list=gene_sets["MAC1_genes"] , score_name="MAC1_score")
sc.pl.umap(adata_snmono, color='MAC1_score', cmap='bwr', title='MAC1_score')

In [ ]:
sc.tl.score_genes(adata_snmono, gene_list=gene_sets["moKC_genes"] , score_name="moKC_score")
sc.pl.umap(adata_snmono, color='moKC_score', cmap='bwr', title='moKC_score')

In [ ]:
sc.tl.score_genes(adata_snmono, gene_list=gene_sets["KC_genes"] , score_name="KC_score")
sc.pl.umap(adata_snmono, color='KC_score', cmap='bwr', title='KC_score')

In [ ]:
sc.tl.leiden(adata_snmono, resolution=0.5, key_added="10_cluster")
sc.pl.umap(adata_snmono, color=['10_cluster'])

In [ ]:
predictions_cluster = celltypist.annotate(adata_snmono, model='Healthy_Human_Liver.pkl', majority_voting=True, over_clustering=adata_snmono.obs["10_cluster"]
)
adata_snmono =predictions_cluster.to_adata()
adata_snmono.obs['celltypist_liver'] = adata_snmono.obs['majority_voting']

In [ ]:
sc.pl.umap(
    adata_snmono,
    color='celltypist_liver',
    #legend_loc='on data',
    legend_fontsize=10,
    frameon=False,
    title='state',
    size=15
)

In [ ]:
predictions_cluster = celltypist.annotate(adata_snmono, model='Immune_All_Low.pkl', majority_voting=True, over_clustering=adata_snmono.obs["10_cluster"]
)
adata_snmono =predictions_cluster.to_adata()
adata_snmono.obs['Immune_low'] = adata_snmono.obs['majority_voting']
sc.pl.umap(
    adata_snmono,
    color='Immune_low',
    #legend_loc='on data',
    legend_fontsize=10,
    frameon=False,
    title='state',
    size=15
)

In [ ]:
predictions_cluster = celltypist.annotate(adata_snmono, model='Immune_All_High.pkl', majority_voting=True, over_clustering=adata_snmono.obs["10_cluster"]
)
adata_snmono =predictions_cluster.to_adata()
adata_snmono.obs['Immune_high'] = adata_snmono.obs['majority_voting']
sc.pl.umap(
    adata_snmono,
    color='Immune_high',
    #legend_loc='on data',
    legend_fontsize=10,
    frameon=False,
    title='state',
    size=15
)

In [ ]:
# 2. crosstab 생성
cluster_labels = adata_snmono.obs['10_cluster']
batch_labels = adata_snmono.obs['state']
crosstab = pd.crosstab(batch_labels, cluster_labels, normalize='index')

# 3. matplotlib barplot 그릴 때 palette 지정
crosstab.plot(
    kind='bar',
    stacked=True,
    figsize=(5, 10),
    #color=[palette_dict[col] for col in crosstab.columns]
)

plt.title('Major cell lineages in MASLD vs Normal')
plt.ylabel('Proportion')
plt.xlabel('state')
plt.legend(title='MASLD vs Normal', bbox_to_anchor=(1.05, 1), loc='upper left')
#plt.savefig("/home/jaehyunchoi/bystander_collaboration/IPF_data/analysis/figures/3publicatlas_celltypist_IPF_state_stackedbargraph.pdf", bbox_inches='tight', dpi=300)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 2))
sc.pl.violin(adata_snmono, keys = 'SLC7A5',groupby='10_cluster',ax=ax)
plt.show()

In [ ]:
from scipy.stats import fisher_exact
import pandas as pd

# 상태 및 클러스터 정보
state = adata_snmono.obs['state']
cluster = adata_snmono.obs['10_cluster']

# 클러스터(세포타입) 목록
clusters = cluster.unique()
results = []

# Donor / IPF 구분
for clus in sorted(clusters):  # 문자형 label일 경우 그냥 sorted() 사용
    # 각 조건별 개수 계산
    donor_in = ((state == "normal") & (cluster == clus)).sum()
    donor_out = ((state == "normal") & (cluster != clus)).sum()
    ipf_in = ((state == "MASLD") & (cluster == clus)).sum()
    ipf_out = ((state == "MASLD") & (cluster != clus)).sum()
    
    table = [[donor_in, donor_out], [ipf_in, ipf_out]]
    oddsratio, pval = fisher_exact(table)
    
    # 비율 계산
    donor_frac = donor_in / (donor_in + donor_out) if (donor_in + donor_out) > 0 else 0
    ipf_frac = ipf_in / (ipf_in + ipf_out) if (ipf_in + ipf_out) > 0 else 0
    
    results.append({
        "cluster": clus,
        "donor_fraction": donor_frac,
        "ipf_fraction": ipf_frac,
        "odds_ratio": oddsratio,
        "p_value": pval
    })

# 결과를 DataFrame으로 정리
df_result = pd.DataFrame(results)

# p-value 기준 정렬 (선택사항)
df_result = df_result.sort_values("cluster")
print(df_result)

import matplotlib.pyplot as plt
import numpy as np

plt.figure(figsize=(15, 6))
x = np.arange(len(df_result))
width = 0.35

# Bar plot
plt.bar(x - width/2, df_result['donor_fraction'], width, label='normal')
plt.bar(x + width/2, df_result['ipf_fraction'], width, label='MASLD')

# X축 설정
plt.xticks(x, df_result['cluster'])
plt.ylabel('Fraction')
plt.title('T cell cluster proportion by condition')
plt.legend()

# p-value 별로 * 표시
for i, pval in enumerate(df_result['p_value']):
    if pval < 0.001:
        stars = '***'
    elif pval < 0.01:
        stars = '**'
    elif pval < 0.05:
        stars = '*'
    else:
        stars = ''
    
    if stars:
        max_height = max(df_result['donor_fraction'][i], df_result['ipf_fraction'][i])
        plt.text(x[i], 0.34, stars, ha='center', va='bottom', fontsize=12, color='black')

plt.tight_layout()
plt.show()

In [ ]:
adata_snmono.write("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/snRNA_v1/snRNA_adata_mono02.h5ad")

In [ ]:
adata_snmono

In [ ]:
sc.pl.umap(adata_snmono, color = '10_cluster')

In [ ]:
Rama_genes ={'Hepatocyte': ['GHR', 'CYP2A7', 'PCK1', 'CYP2A6', 'HSD11B1', 'G6PC', 'BCHE', 'HAMP', 'HPR', 'SEC16B', 'MASP2', 'ACSS2', 'AKR1C1', 'ALDH6A1', 'HMGCS1'], 'Epithelia': ['ALB', 'AGXT', 'TF', 'TTR', 'KRT19', 'SOX9', 'KNG1', 'HSD17B6', 'F2', 'ITIH1', 'CYP4A11'], 'Mesenchyme': ['TIMP1', 'COL3A1', 'ACTA2'], 'Endothelia': ['EGFL7', 'HSPG2', 'OIT3', 'CLEC4G', 'EMCN', 'FCN3', 'CLEC4M', 'CLEC14A'], 'Tcell': ['CD2'], 'ILC': ['FGFBP2', 'PRF1', 'KLRF1', 'KLRC1'], 'B cell': ['CD79A', 'MS4A1'], 'pDC': ['LILRA4', 'PTCRA', 'LRRC26', 'CLEC4C'], 'Plasma cell': ['FCRL5', 'JSRP1'], 'MP': ['FCN1', 'IL1B', 'MS4A7', 'VCAN', 'CPVL', 'MNDA', 'CD163', 'CYBB', 'CSF2RA', 'MSR1', 'CLEC10A', 'C1QC']}

In [ ]:
sc.pl.dotplot(
    adata_snmono,
    Rama_genes,
    groupby="10_cluster",
    standard_scale="var",
    figsize=(11, 8),
    #show=False,
    #return_fig=True
)

In [ ]:
sc.pl.dotplot(adata_snmono, marker_genes, groupby="10_cluster", standard_scale="var")

# Charlotte et al. Marker genes Feature plot

In [ ]:
gene_sets = {
    "Hepa_genes": [ 'CYP2C8', 'CYP3A4', 'CYP2E1', 'CP', 'CYP3A43', 'CUX2', 'SORBS2', 'SDS', 'ZNF385D', 'ERRFI1', 'RNF152', 'SLC27A2', 'SLC16A12', 'C3', 'F5', 'GREM2', 'MAOB', 'CYP3A5', 'AASS', 'LIPC', 'NOS1AP', 'UPB1', 'GPLD1', 'ARG1', 'TENM1', 'FAM9B', 'COBLL1', 'ALDOB', 'RP11-432I5.1', 'FMO5', 'GPAM', 'CDHR3', 'MARC2', 'IGFBP1', 'GPR126', 'OSBPL6', 'NEK10', 'CYP4F3', 'PLIN5', 'SLC1A2', 'TMEM150C', 'CXADR', 'IGFBP2', 'CMYA5', 'TFR2', 'NECAB2', 'ADAMTS17', 'NRG1', 'FGA', 'RP11-431K24.1'],
    "Chola_genes": ['DCDC2', 'SCGN', 'CFTR', 'BICC1', 'ITGB8', 'KRT19', 'CTNND2', 'CLDN10', 'KRT7', 'SNAP25', 'DEFB1', 'MUC6', 'CASR', 'FAM150B', 'SEMA3E', 'PKHD1', 'FXYD2', 'CCDC64B', 'ELF3', 'HUNK', 'STK33', 'CASC15', 'KCNJ15', 'FGFR2', 'KRT8', 'CALN1', 'STXBP6', 'EPB41L1', 'LINC00671', 'PRDM16', 'SLC38A11', 'ANXA3', 'SEMA6D', 'C1orf186', 'ATP13A4', 'MACC1', 'SLC35F3', 'FLRT2', 'LINC00511', 'LGALS4', 'TACSTD2', 'KLF5', 'PMEPA1', 'ANXA4', 'SCD5', 'MAP2', 'BACE2', 'FAM171A1', 'CNTN4', 'TTN'],
    "cDC2_genes": ['FCER1A', 'CD1C', 'CD1E', 'CLEC10A', 'PPP1R14A', 'HLA-DQA1', 'HLA-DRA', 'CSF2RA', 'PKIB', 'PLD4', 'HLA-DQB1', 'HLA-DPB1', 'IL1R2', 'LGALS2', 'CD74', 'RNASE6', 'HLA-DRB5', 'HLA-DPA1', 'HLA-DRB1', 'CST3', 'CLIC2', 'SDPR', 'PHACTR1', 'FILIP1L', 'P2RY14', 'IFI30', 'LY86', 'IL1B', 'CD86', 'GAPT', 'C15orf48', 'HLA-DMB', 'LYZ', 'FCGR2B', 'CXorf21', 'ADAM28', 'IL18', 'SGK1', 'IL13RA1', 'GPR183', 'CPVL', 'HLA-DMA', 'KLF4', 'RAB32', 'NR4A3', 'CFP', 'IGSF6', 'PAK1', 'AXL', 'PLBD1'],
    "cDC_genes":['CCL19', 'CCL22', 'LAMP3', 'LAD1', 'CCR7', 'FSCN1', 'IDO1', 'CCL17', 'EBI3', 'CD1E', 'C15orf48', 'AC079767.4', 'DUSP4', 'CD200', 'CSF2RA', 'MARCKSL1', 'IL4I1', 'RAMP1', 'PTGIR', 'CLIC2', 'MGLL', 'SPIB', 'BIRC3', 'DAPP1', 'CD83', 'HLA-DOB', 'TXN', 'HLA-DQA1', 'NRP2', 'CD1C', 'HLA-DRA', 'C12orf45', 'MARCKS', 'FABP5', 'GSN', 'LGALS2', 'MIR155HG', 'SERPINF1', 'HILPDA', 'RASSF4', 'HLA-DQB1', 'CTSH', 'CLECL1', 'CD74', 'HLA-DPB1', 'BASP1', 'TNFAIP2', 'GPR183', 'FILIP1L', 'NR4A3'],
    "Mono_genes":['CD300E', 'FCN1', 'RP11-290F20.3', 'EREG', 'LILRA5', 'APOBEC3A', 'LILRB2', 'CSTA', 'VCAN', 'IFI30', 'FCGR1A', 'THBS1', 'IL1B', 'SLC7A7', 'LRRC25', 'AIF1', 'SLC11A1', 'HCK', 'SPI1', 'HBEGF', 'LYZ', 'PRAM1', 'MAFB', 'NCF2', 'LST1', 'RP11-1143G9.4', 'LGALS2', 'PILRA', 'CLEC7A', 'PLAUR', 'THBD', 'CYBB', 'CLEC4E', 'CFD', 'CD86', 'EMR2', 'S100A9', 'CLEC12A', 'KCTD12', 'PLBD1', 'SERPINA1', 'KLF4', 'C5AR1', 'CD68', 'BTK', 'TLR2', 'IFITM3', 'CSF1R', 'FCGR2A', 'FPR2'],
    "Neutrophil_genes":['FCGR3B', 'CMTM2', 'S100P', 'PTGS2', 'CXCR2', 'S100A8', 'G0S2', 'S100A12', 'CSF3R', 'CTB-61M7.2', 'MNDA', 'IL8', 'S100A9', 'FPR1', 'IL1R2', 'FPR2', 'MXD1', 'TREM1', 'VNN2', 'BASP1', 'LILRB3', 'GCA', 'NAMPT', 'CDA', 'PLBD1', 'LRRK2', 'SELL', 'CLEC4E', 'NCF2', 'C5AR1', 'RGS18', 'TRIB1', 'CD93', 'RBP7', 'SLC25A37', 'TMEM154', 'EGR1', 'AQP9', 'SLC11A1', 'VCAN', 'TLR2', 'CSTA', 'MEGF9', 'PLAUR', 'SOD2', 'RGS2', 'APOBEC3A', 'IRAK3', 'LYZ', 'RP11-1143G9.4'],
    "Basophil_genes":['TPSAB1', 'CMA1', 'CPA3', 'RP11-354E11.2', 'CTSG', 'MS4A2', 'GATA2', 'HPGDS', 'GSTM5', 'HDC', 'KIT', 'RGS13', 'KRT1', 'GCSAML', 'SLC18A2', 'IL1RL1', 'VWA5A', 'C1orf186', 'CNRIP1', 'LTC4S', 'HPGD', 'STXBP6', 'SLC45A3', 'BACE2', 'GSTM3', 'CD9', 'FCER1A', 'SDPR', 'CTD-3203P2.2', 'RAB27B', 'CLU', 'ACOT7', 'TIMP3', 'ALOX5', 'LMNA', 'SOX4', 'SLC26A2', 'BTK', 'CAPG', 'RP11-620J15.3', 'TSC22D1', 'RAB32', 'STX3', 'MAOB', 'STMN1', 'AREG', 'CPM', 'CD82', 'SERPINB1', 'LMO4'],
    "NK_genes":['XCL1', 'KLRC1', 'IL2RB', 'NCR1', 'TOX2', 'SPRY2', 'CD160', 'XCL2', 'SH2D1B', 'KLRF1', 'TXK', 'CCL3', 'SLFN13', 'TMIGD2', 'CLIC3', 'NCAM1', 'CD38', 'P2RY11', 'EOMES', 'MCTP2', 'CD7', 'CMC1', 'SPRY1', 'LAT2', 'FASLG', 'PTGDR', 'AREG', 'TIGIT', 'MATK', 'TOX', 'CD247', 'ABCB1', 'KLRD1', 'CCL4', 'CXCR6', 'S1PR5', 'RP11-222K16.2', 'IFITM1', 'COG2', 'APBA2', 'RP11-489E7.4', 'KLRB1', 'IL18', 'NKG7', 'IRF8', 'FCER1G', 'CXXC5', 'IFNG', 'SH2D1A', 'SYK'],
    "T_genes":['IL7R', 'CD40LG', 'CD3D', 'CD3G', 'CD8B', 'TRAT1', 'CD8A', 'SIT1', 'GPR171', 'CD3E', 'BCL11B', 'CD6', 'GZMK', 'LAG3', 'SPOCK2', 'ITM2A', 'IL32', 'CD2', 'AC092580.4', 'CXCR6', 'KLRG1', 'TC2N', 'CD27', 'LTB', 'ITK', 'CD69', 'RASGRP1', 'CCL5', 'KLRB1', 'GZMH', 'SH2D1A', 'RHOH', 'NCR3', 'SCML4', 'GZMA', 'RP11-138A9.2', 'MYBL1', 'LAT', 'RP11-94L15.2', 'CD96', 'PRR5', 'SH2D2A', 'DUSP2', 'GZMM', 'TUBA4A', 'ODF2L', 'CST7', 'PTPN7', 'LYAR', 'LBH'],
    "pDC_genes":['SCT', 'LRRC26', 'LILRA4', 'MAP1A', 'PTCRA', 'LAMP5', 'CLEC4C', 'MYBL2', 'SMIM5', 'KCNK17', 'PACSIN1', 'RP11-117D22.2', 'TCL1A', 'IL3RA', 'SPIB', 'PLD4', 'BCL11A', 'RASD1', 'FAM129C', 'TPM2', 'TSPAN13', 'PTPRS', 'SERPINF1', 'PPP1R14A', 'DERL3', 'SMPD3', 'IRF4', 'ITM2C', 'PFKFB2', 'PTGDS', 'FUT7', 'SOX4', 'IRF7', 'CXCR3', 'P2RY14', 'NOTCH4', 'PMEPA1', 'LINC00996', 'PPP1R14B', 'CLN8', 'VIPR2', 'AEBP1', 'C1orf186', 'MZB1', 'IRF8', 'TCF4', 'UGCG', 'GZMB', 'TNFRSF21', 'GAPT'],
    "B_genes":['VPREB3', 'FCRL1', 'RP11-693J15.5', 'LINC00926', 'FCRLA', 'FCRL2', 'CD19', 'CD79A', 'PAX5', 'RP5-887A10.1', 'AC079767.4', 'MS4A1', 'BANK1', 'CTA-250D10.23', 'CD22', 'TNFRSF13B', 'TNFRSF13C', 'KIAA0125', 'FCRL5', 'FCER2', 'TCL1A', 'FAM129C', 'POU2AF1', 'HLA-DOB', 'BLK', 'SPIB', 'CD79B', 'ADAM28', 'CCR7', 'BCL11A', 'BTLA', 'ARHGAP24', 'HVCN1', 'MARCH1', 'GNG7', 'CD72', 'IFT57', 'RIC3', 'SWAP70', 'P2RX5', 'MEF2C', 'BTK', 'GAPT', 'BLNK', 'DAPP1', 'RALGPS2', 'AFF3', 'CXXC5', 'SELL', 'FCGR2B'],
    "Plasma_genes":['RP11-731F5.2', 'IGJ', 'IGLL5', 'TNFRSF17', 'RP11-16E12.2', 'MZB1', 'DERL3', 'AL928768.3', 'POU2AF1', 'FCRL5', 'AC104024.1', 'SPAG4', 'JSRP1', 'KIAA0125', 'TNFRSF13B', 'ABCB9', 'BHLHE41', 'SSPN', 'CCR10', 'PNOC', 'AC006129.4', 'RP5-887A10.1', 'CD79A', 'SDC1', 'ITM2C', 'PRDX4', 'CAV1', 'AC093818.1', 'SEC11C', 'HLA-DOB', 'GNG7', 'SSR4', 'RAB30', 'TNFRSF13C', 'HBB', 'PPAPDC1B', 'HERPUD1', 'FKBP11', 'CD38', 'QPCT', 'CD27', 'XBP1', 'C16orf74', 'HSP90B1', 'PDK1', 'PIM2', 'MEI1', 'FAM46C', 'SEL1L3', 'LRRC16A'],
    "cDC1_genes":['CLEC9A', 'TACSTD2', 'IDO1', 'WDFY4', 'C1orf54', 'HLA-DOB', 'CLNK', 'CPVL', 'EGLN3', 'DNASE1L3', 'KIAA0226L', 'LGALS2', 'CST3', 'CCND1', 'S100B', 'HLA-DPB1', 'CD74', 'DUSP4', 'HLA-DRA', 'HLA-DPA1', 'HLA-DQB1', 'C15orf48', 'HLA-DQA1', 'HLA-DRB5', 'SNX3', 'RAB32', 'P2RY14', 'HLA-DRB1', 'RGCC', 'CSF2RA', 'IRF8', 'SERPINF1', 'BTLA', 'LMNA', 'ADAM28', 'TSPAN13', 'ASB2', 'CLIC2', 'AIM2', 'KCNK6', 'BASP1', 'MPEG1', 'C12orf45', 'LYZ', 'CXCL16', 'RGS10', 'VMO1', 'DAPP1', 'LY86', 'SEMA4A'],
    "MAC_genes":['NDST3', 'CD5L', 'LILRB5', 'SDC3', 'FOLR2', 'C1QC', 'C1QB', 'CTD-2337J16.1', 'SIGLEC1', 'C1QA', 'VCAM1', 'MARCO', 'TIMD4', 'AXL', 'CXCL12', 'CETP', 'IGSF21', 'LYVE1', 'GFRA2', 'SLC40A1', 'GPNMB', 'APOE', 'LGMN', 'MSR1', 'GPR34', 'MRC1', 'VSIG4', 'PLTP', 'FPR3', 'CD163', 'EBI3', 'SEPP1', 'ADORA3', 'MS4A4A', 'MRC1L1', 'ETV5', 'SLC1A3', 'RND3', 'TNFRSF21', 'PLA2G7', 'ADAP2', 'RGL1', 'C2', 'FMNL2', 'FILIP1L', 'MERTK', 'DAB2', 'CR1', 'MS4A7', 'CMKLR1']}


In [ ]:
sc.tl.score_genes(adata_snrna, gene_list=gene_sets["Mono_genes"] , score_name="Mono_score")
sc.pl.umap(adata_snrna, color='Mono_score', cmap='bwr', title='Mono_score')

In [ ]:
sc.tl.score_genes(adata_snrna, gene_list=gene_sets["Plasma_genes"] , score_name="Plasma_score")
sc.pl.umap(adata_snrna, color='Plasma_score', cmap='bwr', title='Plasma_score')

In [ ]:
sc.tl.score_genes(adata_snrna, gene_list=gene_sets["MAC_genes"] , score_name="MAC_score")
sc.pl.umap(adata_snrna, color='MAC_score', cmap='bwr', title='MAC_score')

In [ ]:
sc.tl.score_genes(adata_snrna, gene_list=gene_sets["Hepa_genes"] , score_name="Hepa_score")
sc.pl.umap(adata_snrna, color='Hepa_score', cmap='bwr', title='Hepa_score')

In [ ]:
adata_snmono = adata_snraw[adata_snraw.obs['cluster'] == '6'].copy()
adata_snmono

In [ ]:
sc.pp.normalize_total(adata_snmono, target_sum=1e4)
sc.pp.log1p(adata_snmono)
# 변수 유전자 설정 
sc.pp.highly_variable_genes(adata_snmono, flavor='seurat', n_top_genes=2000)
# Scale data
sc.pp.scale(adata_snmono, max_value=10)
# Run PCA
sc.tl.pca(adata_snmono, svd_solver='arpack')
print(adata_snmono.obsm['X_pca'][:5, :5])
sc.pl.pca(adata_snmono, color='sample')

import matplotlib.pyplot as plt
var_ratio = adata_snmono.uns['pca']['variance_ratio']
plt.plot(range(1, len(var_ratio) + 1), np.cumsum(var_ratio), marker='o')
plt.xlabel('Number of Principal Components')
plt.ylabel('Cumulative Explained Variance')
plt.title('PCA Cumulative Variance')
plt.grid(True)
plt.show()

In [ ]:
# t-sne 
sc.tl.tsne(adata_snmono, use_rep='X_pca', n_pcs=30)
# umap
sc.pp.neighbors(adata_snmono, n_pcs=30, use_rep='X_pca')
sc.tl.umap(adata_snmono)
sc.pl.umap(adata_snmono, color=['sample'])

In [ ]:
adata_snmono.write("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/snRNA_v1/snRNA_mono_01.h5ad")

In [ ]:
sc.pl.umap(adata_snmono, color=['state'])

In [ ]:
sc.pl.umap(adata_snmono, color=['PTPRC'])

In [ ]:
sc.pl.umap(adata_snmono, color=['SLC7A5'])

In [ ]:
gene_sets = {
    "cDC1_genes": ['XCR1', 'CLNK', 'TACSTD2', 'CLEC9A', 'EGLN3', 'IDO1', 'DNASE1L3', 'CADM1', 'S100B', 'KIAA0226L', 'WDFY4', 'CCND1', 'DUSP4', 'P2RY14', 'C1orf54', 'NRARP', 'CPNE3', 'SLAMF7', 'GPR157', 'B3GNT7', 'FLT3', 'NET1', 'RGCC', 'SERPINF1', 'BATF3', 'NDRG2', 'SNX3', 'ADAM28', 'AIM2', 'IL18R1', 'GPR171', 'PTPRCAP', 'IRF8', 'CPVL', 'C12orf45', 'TPD52', 'SLC38A1', 'LMNA', 'CD59', 'C15orf48', 'NFATC2', 'NAV1', 'MAP2K6', 'PTMS', 'ARHGAP5', 'HLA-DPB1', 'CST7', 'LGALS2', 'HLA-DQB1', 'PPA1', 'TCTN3', 'APOL1', 'CAMK2D', 'HLA-DPA1', 'CD74', 'HLA-DQA1', 'CST3', 'BCL2L11', 'DHRS3', 'ID2', 'RAB11FIP1', 'QPRT', 'OSBPL9', 'PTPN22', 'PTPN7', 'FAM129A', 'DST', 'FUCA1', 'ANPEP', 'MARCKSL1', 'HLA-DRB1', 'CLIC2', 'STK17A', 'RGS1', 'HLA-DRA', 'BASP1', 'ACTN1', 'PDLIM7', 'CD40', 'AC093673.5', 'NCOA7', 'PDE4DIP', 'TUBA1C', 'STMN1', 'CSF2RA', 'CD38', 'MIR4435-1HG', 'DUSP2', 'LGMN', 'PKIB', 'VMO1', 'SPATS2L'],
    "cDC2_genes": ['FCER1A', 'PPP1R14A', 'CD1C', 'AKR1C3', 'ENHO', 'CD1E', 'SLC38A1', 'P2RY14', 'AFF3', 'FCGR2B', 'CLEC10A', 'NDRG2', 'CST7', 'PKIB', 'CACNA2D3', 'ADAM28', 'SDPR', 'PLD4', 'NET1', 'FLT3', 'GAS6', 'ALOX5AP', 'AGPAT9', 'HLA-DQA1', 'RGS1', 'FILIP1L', 'C12orf45', 'HLA-DQB1', 'IL1R2', 'ARHGAP5', 'HLA-DPB1', 'GPR183', 'AXL', 'ADAM8', 'USP53', 'SERPINF1', 'RGCC', 'HLA-DPA1', 'CD72', 'GSN', 'CLIC2', 'ACAP1', 'ATP1B1', 'HLA-DRA', 'PMAIP1', 'CD74', 'CSF2RA', 'AREG', 'PPA1', 'FPR3', 'HLA-DRB1', 'ALCAM', 'SPATS2L', 'ARL4C', 'PTMS', 'STMN1', 'LMNA', 'VSIG4'],
    "pDC_genes":['SCT', 'LRRC26', 'LILRA4', 'MAP1A', 'PTCRA', 'LAMP5', 'CLEC4C', 'MYBL2', 'SMIM5', 'KCNK17', 'PACSIN1', 'RP11-117D22.2', 'TCL1A', 'IL3RA', 'SPIB', 'PLD4', 'BCL11A', 'RASD1', 'FAM129C', 'TPM2', 'TSPAN13', 'PTPRS', 'SERPINF1', 'PPP1R14A', 'DERL3', 'SMPD3', 'IRF4', 'ITM2C', 'PFKFB2', 'PTGDS', 'FUT7', 'SOX4', 'IRF7', 'CXCR3', 'P2RY14', 'NOTCH4', 'PMEPA1', 'LINC00996', 'PPP1R14B', 'CLN8', 'VIPR2', 'AEBP1', 'C1orf186', 'MZB1', 'IRF8', 'TCF4', 'UGCG', 'GZMB', 'TNFRSF21', 'GAPT'],
    "MigcDC_genes": ['LAD1', 'CCR7', 'LAMP3', 'CCL19', 'CCL22', 'IL7R', 'SLC7A11', 'FSCN1', 'RAMP1', 'IDO1', 'CCL17', 'CD1B', 'MIR155HG', 'TNFRSF4', 'CXCL9', 'GPR157', 'DUSP4', 'IL4I1', 'EBI3', 'TRAF1', 'MARCKSL1', 'SPIB', 'ARHGAP10', 'HLA-DQB2', 'CD1E', 'MGLL', 'IL32', 'DUSP5', 'NRP2', 'CST7', 'PSD3', 'BIRC3', 'IRF4', 'C15orf48', 'HILPDA', 'GADD45A', 'SLC38A1', 'TRAF5', 'C12orf45', 'TXN', 'RAB9A', 'TBC1D4', 'TUBA1C', 'RGS1', 'SERPINF1', 'NMRK1', 'PPA1', 'CD40', 'GSN', 'POGLUT1', 'CLIC2', 'NAV1', 'PVRL2', 'PTGIR', 'RP11-138A9.2', 'FABP5', 'ALCAM', 'FAM129A', 'NET1', 'CSF2RA', 'ANTXR2', 'UGCG', 'GPR137B', 'CD1C', 'ADAM8', 'RELB', 'PALLD', 'SLAMF7', 'ST3GAL6', 'RP11-356I2.4', 'GPR183', 'HLA-DQA1', 'NFKB1', 'FLT3', 'RP11-138A9.1', 'CERS6', 'DNASE1L3', 'P2RY8', 'CD83', 'MAFF', 'SESN1', 'CYB5A', 'PTMS', 'SDC2', 'CCL5', 'ATP1B1', 'ETV3', 'ATF5', 'HLA-DQB1', 'FILIP1L', 'PMAIP1', 'PIM3', 'ID2', 'MARCKS', 'EPSTI1', 'BHLHE40', 'SOCS1', 'BASP1', 'HLA-DPB1', 'CD74', 'APOC1', 'HLA-DRA', 'CD72', 'GBP1', 'NR4A3', 'HLA-DPA1', 'PLEK', 'LMNA', 'CLEC10A'],
    "Monocyte_genes":['S100A8', 'S100A12', 'S100A9', 'VCAN', 'CTB-61M7.2', 'THBS1', 'EREG', 'MGST1', 'RP11-1143G9.4', 'ASGR1', 'BST1', 'AQP9', 'FCN1', 'CDA', 'LYZ', 'SLC2A3', 'MEGF9', 'APOBEC3A', 'CD300E', 'CD36', 'RAB27A', 'CLEC4E', 'TMEM176A', 'SELL', 'LGALS2', 'CXCL2', 'ITGAM', 'CSF3R', 'G0S2', 'EGR1', 'SOCS3', 'CCDC69', 'CAPG', 'TREM1', 'SLC11A1', 'AREG'],
    "Patmono_genes":['LYPD2', 'CDKN1C', 'CKB', 'PTP4A3', 'CD79B', 'HES4', 'PPM1N', 'SYTL1', 'FCGR3A', 'LILRA3', 'RHOC', 'RP11-290F20.3', 'TESC', 'SPN', 'TCF7L2', 'TPPP3', 'SLC2A6', 'SLC44A2', 'RRAS', 'SNX18', 'OAS1', 'FAM110A', 'VMO1', 'MTSS1', 'LILRB2', 'PHF19', 'TBC1D10C', 'FPR2', 'LILRA5', 'IFITM2', 'SIGLEC10', 'MYO1G', 'PAG1', 'SIDT2', 'LTB', 'LST1', 'S1PR4', 'ITGAL', 'STXBP2', 'CNIH4', 'GCH1', 'IFITM3', 'KLF2'],
    "LAM_genes":['FN1', 'CST6', 'SPP1', 'MT1G', 'TREM2', 'GPNMB', 'SLC16A10', 'CD9', 'EMP1', 'APOC1', 'LMNA', 'C15orf48', 'CLEC5A', 'IL7R', 'FABP5', 'RNASE1', 'FBP1', 'SDS', 'RAB13', 'OLR1', 'APOE', 'RGCC', 'GPX3', 'TSC22D1', 'SLC39A8', 'CAPG', 'ACP5', 'MMP19', 'ALCAM', 'GSN', 'FPR3', 'PDE4DIP', 'MT1X', 'LHFPL2', 'SDC2', 'MITF', 'PLA2G7', 'LGALS3', 'PHLDA1', 'ANKRD28', 'AVPI1', 'PTMS', 'MARCO', 'B3GNT5', 'PLTP', 'GPR183', 'DAB2', 'CD59', 'RGS1', 'LGMN'],
    "MATLAM_genes":['TREM2', 'GPNMB', 'CD9', 'RNASE1', 'OTOA', 'SDS', 'A2M', 'APOC1', 'LGMN', 'FCGBP', 'FABP5', 'SLCO2B1', 'APOE', 'C1QC', 'PLA2G7', 'FPR3', 'ID3', 'DAB2', 'AXL', 'C1QB', 'PLTP', 'RGS1', 'LTC4S', 'CD72', 'GATM', 'MGLL', 'LMNA', 'ABCC5', 'GAS6', 'ANKH', 'STMN1', 'C1QA', 'ALCAM', 'MSR1', 'PKIB', 'PHLDA1', 'ATP1B1', 'GPR34', 'PTMS', 'RGCC', 'MGAT4A', 'FCGR2B', 'LIPA', 'ACP5', 'CTSL', 'MITF', 'MAF', 'EBI3', 'KCNMA1', 'MERTK', 'VSIG4', 'C15orf48', 'FILIP1L', 'SEPP1', 'SLC40A1', 'CD81', 'GPR183', 'SDC2', 'GSN', 'FOLR2', 'CD59', 'CD84', 'CPM', 'MS4A4A', 'ABCA1', 'HLA-DQA1', 'CLEC10A', 'CLIC2', 'ARL4C', 'RGL1', 'EPB41L2', 'MMP19', 'AVPI1', 'ALB', 'C1orf54', 'PDK4', 'CD1C', 'VMO1'],
    "MAC1_genes":['IGSF21', 'FCGBP', 'USP53', 'ID3', 'C1QC', 'A2M', 'SDS', 'PLTP', 'MRC1L1', 'SLCO2B1', 'LTC4S', 'CD9', 'FCGR2B', 'RGS1', 'VSIG4', 'FPR3', 'C1QB', 'GPR34', 'AXL', 'FILIP1L', 'PKIB', 'C1QA', 'CD72', 'CLEC9A', 'NDRG2', 'CLEC10A', 'GATM', 'ALB', 'GPR183', 'MSR1', 'HLA-DPA1', 'PAPOLG', 'DAB2', 'PLD4', 'LPAR6', 'PHLDA1', 'EBI3', 'F13A1', 'FCER1A', 'HLA-DQA1', 'CPM', 'HLA-DPB1', 'ALOX5AP', 'SERPINF1', 'HLA-DRB1', 'RCAN1', 'STMN1', 'HLA-DRA', 'CCL3L1', 'ATP1B1', 'HLA-DQB1', 'PRDM1', 'MAF', 'ALCAM', 'C1orf54', 'APOC1', 'PTMS', 'LGALS3BP', 'CLIC2', 'RGCC', 'SLC40A1', 'VMO1', 'SERPING1', 'EPB41L2', 'CCL3', 'FOLR2', 'CD1C', 'PDK4', 'SEPP1', 'APOE'],
    "moKC_genes":['RNASE1', 'IGSF21', 'FCGBP', 'LILRB5', 'SLCO2B1', 'C1QC', 'SEPP1', 'PLTP', 'MRC1L1', 'NRP1', 'A2M', 'C1QB', 'TMEM37', 'LYVE1', 'FRMD4A', 'GFRA2', 'APOE', 'FOLR2', 'GPR34', 'OLFML2B', 'C1QA', 'LGMN', 'MAF', 'ID3', 'CXCL12', 'AXL', 'MERTK', 'SLC40A1', 'DAB2', 'MRC1', 'DHRS3', 'PDK4', 'GATM', 'LTC4S', 'MS4A4A', 'GPNMB', 'APOC1', 'USP53', 'EBI3', 'FPR3', 'FILIP1L', 'ALB', 'STAB1', 'SDC3', 'SIGLEC1', 'KCNMA1', 'VSIG4', 'ANKH', 'PKIB', 'MSR1', 'VCAM1', 'FCHO2', 'CCL3', 'SERPING1', 'CLEC9A', 'LGALS3BP', 'CCL4', 'CCL3L1', 'CD72', 'STMN1', 'CPM', 'CD163', 'RGL1', 'ABCA1', 'CD5L', 'EPS8', 'C2', 'CLIC2', 'RGS1', 'SPRED1', 'ATP1B1', 'TSPAN4', 'RCAN1', 'LIPA', 'SMPDL3A', 'WWP1', 'FRMD4B', 'EPB41L2', 'CD81', 'TCF4', 'AP2A2', 'SPATS2L', 'CD84', 'DST', 'FMNL2', 'MS4A7', 'CD59', 'CTSL', 'PLD3', 'HLA-DPA1', 'PTMS', 'ARL4C', 'VMO1', 'C1orf54', 'EGR2', 'CLEC10A', 'FCGR3A'],
    "KC_genes":['TIMD4', 'SLC16A9', 'NDST3', 'KCNAB1', 'ITGAD', 'CD5L', 'VCAM1', 'SELENBP1', 'BCAM', 'CDH5', 'CETP', 'RND3', 'FEZ1', 'FABP3', 'SDC3', 'CXCL12', 'ITLN1', 'CTD-2337J16.1', 'LYVE1', 'SEPP1', 'TMEM37', 'MARCO', 'SCD', 'LILRB5', 'PPAP2B', 'ITGA9', 'IGF1', 'APOE', 'SLC7A8', 'EPAS1', 'SLC40A1', 'FRMD4A', 'SUCNR1', 'ACVRL1', 'IGFBP4', 'ME1', 'LGMN', 'SIGLEC1', 'SIGLEC11', 'GPR126', 'FOLR2', 'LAG3', 'ITGB5', 'NRP1', 'C1QB', 'AXL', 'MRC1', 'C1QA', 'IFI27', 'C1QC', 'C2', 'SLC46A1', 'SLCO2B1', 'ETV5', 'SCN1B', 'NR1H3', 'DHRS3', 'PDK4', 'CD209', 'CCL4L1', 'GPNMB', 'ARHGEF12', 'CD59', 'RGL1', 'ANKRD36C', 'CCL4', 'MAF', 'EPB41L2', 'GATM', 'CCND1', 'CTSF', 'IFT74', 'LIPA', 'A2M', 'MYO9A', 'ATP1B1', 'CCL4L2', 'IFIT1', 'SERPING1', 'SPATS2L', 'MERTK', 'FUCA1', 'ADORA3', 'SMPDL3A', 'CCL3', 'ABCA1', 'MRC1L1', 'ANKH', 'CMKLR1', 'DST', 'PLD3', 'PLTP', 'FSCN1', 'APOC1', 'MMD', 'QPRT', 'VSIG4', 'RHOBTB3', 'EXT1', 'DAB2', 'ALB', 'RCAN1', 'FAM213A', 'EBI3', 'GFRA2', 'CD82', 'CCL3L3', 'PLA2G15', 'WWP1', 'CD81', 'ADM', 'ABCG1', 'FXYD6', 'STOM', 'KCNMA1', 'IFIT2', 'CD38', 'PPP1R12B', 'APOC3', 'CTSL', 'LGALS3BP', 'CR1', 'MSR1', 'TNFRSF21', 'CD163', 'GCNT1', 'EPHX1', 'SASH1', 'FILIP1L', 'SNX24', 'CLIC2', 'IFIT3', 'C1orf54', 'FPR3', 'FRMD4B', 'LTC4S', 'MMP19', 'CREG1', 'HMOX1', 'ZNF189', 'SDPR', 'NCEH1', 'FMNL2', 'CCL3L1', 'ITGAV', 'TSPAN4', 'ALDH1A1', 'NFIA', 'CTSB', 'PDE4DIP', 'ARL4C', 'ARRDC3', 'STMN1', 'RBP7', 'MS4A4A', 'EGR2', 'CD84', 'TCF4', 'VMO1', 'DSC2', 'GPR34', 'FCHO2', 'SLC9A9', 'PLA2G7', 'ADAP2', 'SCARB2', 'IL10', 'FCGRT', 'PHACTR2', 'AP2A2', 'ELL2', 'ACP2', 'MS4A7', 'P4HA1', 'PDGFC', 'GPR137B', 'SGPP1', 'SPRED1', 'PKIB', 'IFI44L', 'PRNP', 'CD72', 'SIAH2', 'MS4A6A', 'EPS8', 'UGCG', 'IDH1', 'BLVRB', 'ICAM1', 'RGS1', 'CTSD', 'CD63', 'FCGR3A', 'MT1F', 'GIMAP5', 'KCNJ2']
    }


In [ ]:
sc.tl.score_genes(adata_snmono, gene_list=gene_sets["KC_genes"] , score_name="KC_score")
sc.pl.umap(adata_snmono, color='KC_score', cmap='bwr', title='KC_score')

In [ ]:
sc.tl.score_genes(adata_snmono, gene_list=gene_sets["Monocyte_genes"] , score_name="Monocyte_score")
sc.pl.umap(adata_snmono, color='Monocyte_score', cmap='bwr', title='Monocyte_score')

In [ ]:
sc.tl.score_genes(adata_snmono, gene_list=gene_sets["MAC1_genes"] , score_name="MAC1_score")
sc.pl.umap(adata_snmono, color='MAC1_score', cmap='bwr', title='MAC1_score')

In [ ]:
sc.tl.score_genes(adata_snmono, gene_list=gene_sets["MigcDC_genes"] , score_name="MigcDC_genes_score")
sc.pl.umap(adata_snmono, color='MigcDC_genes_score', cmap='bwr', title='MigcDC_genes_score')

# scvi - workflow

In [ ]:
import torch
import scvi
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scanpy as sc
import pandas as pd

In [ ]:
print(torch.cuda.is_available())
print("cuda available:", torch.cuda.is_available())
print("cuda device count:", torch.cuda.device_count())
print("device name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No CUDA device")
print("torch version:", torch.__version__)
print("cuda version:", torch.version.cuda)

In [ ]:
scvi.settings.seed = 0
print("Last run with scvi-tools version:", scvi.__version__)

In [ ]:
adata_snraw=sc.read("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/snRNA_v1/snRNA_raw.h5ad")
#adata_snrna=sc.read("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/snRNA_v1/snRNA_adata_01.h5ad")

In [ ]:
adata_snraw.obs_names_make_unique

In [ ]:
adata_snraw.obs['state']

In [ ]:
adata_snraw.layers["counts"] = adata_snraw.X.copy()  # preserve counts
sc.pp.normalize_total(adata_snraw, target_sum=1e4)
sc.pp.log1p(adata_snraw)
adata_snraw.raw = adata_snraw

In [ ]:
sc.pp.highly_variable_genes(
    adata_snraw,
    n_top_genes=2000,
    #subset=True,
    layer="counts",
    flavor="seurat_v3",
)

In [ ]:
scvi.model.SCVI.setup_anndata(
    adata_snraw,
    layer="counts",
    categorical_covariate_keys=["sample"],
    continuous_covariate_keys=["mt_frac", "ribo_frac"],
)

In [ ]:
model = scvi.model.SCVI(adata_snraw)
model

In [ ]:
model.train()

In [ ]:
model.save("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/snRNA_v1/scvi/", overwrite = True)

In [ ]:
SCVI_LATENT_KEY = "X_scVI"
latent = model.get_latent_representation()

In [ ]:
adata_snraw.obsm[SCVI_LATENT_KEY] = latent
latent.shape

In [ ]:
sc.pp.neighbors(adata_snraw, use_rep=SCVI_LATENT_KEY)
sc.tl.umap(adata_snraw, min_dist=0.3)

In [ ]:
adata_snraw.obs_names_make_unique()

In [ ]:
adata_snraw

In [ ]:
sc.tl.leiden(adata_snraw)

In [ ]:
sc.pl.umap(
    adata_snraw,
    color=["batch", "leiden"],
    frameon=False,
    ncols=1,
)

In [ ]:
adata_snraw.write("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/snRNA_v1/snRNA_scvi_01.h5ad")

In [ ]:
adata_snraw=sc.read("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/snRNA_v1/snRNA_scvi_01.h5ad")

In [ ]:
sc.pl.umap(
    adata_snraw,
    color=["batch", "leiden"],
    frameon=False,
    ncols=1,
)

In [ ]:
celltypist.models.models_description()

In [ ]:
predictions_cluster = celltypist.annotate(adata_snraw, model='Healthy_Human_Liver.pkl', majority_voting=True, over_clustering=adata_snraw.obs["leiden"]
)

In [ ]:
adata_snraw =predictions_cluster.to_adata()

In [ ]:
adata_snraw.obs

In [ ]:
sc.pl.umap(
    adata_snraw,
    color=["majority_voting","leiden"],
    frameon=False,
)

In [ ]:
adata_snraw.write("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/snRNA_v1/snRNA_scvi_01.h5ad")

# sc_anvi analysis

In [ ]:
adata_snraw=sc.read("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/snRNA_v1/snRNA_scvi_01.h5ad")

In [ ]:
adata_snraw

In [ ]:
adata_snraw.obs['majority_voting'].value_counts()

In [ ]:
adata_snraw.obs['cell_type'] = adata_snraw.obs['majority_voting']

In [ ]:
sc.pl.umap(
    adata_snraw,
    color=["cell_type","leiden"],
    frameon=False,
)

In [ ]:
from scvi.model import SCVI

# 저장된 모델 불러오기
model = SCVI.load("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/snRNA_v1/scvi/", adata=adata_snraw)

In [ ]:
scanvi_model = scvi.model.SCANVI.from_scvi_model(
    model,
    adata=adata_snraw,
    labels_key="cell_type",
    unlabeled_category="Unknown",
)

In [ ]:
scanvi_model.train(max_epochs=20, n_samples_per_label=100)

In [ ]:
SCANVI_LATENT_KEY = "X_scANVI"
adata_snraw.obsm[SCANVI_LATENT_KEY] = scanvi_model.get_latent_representation(adata_snraw)

In [ ]:
sc.pp.neighbors(adata_snraw, use_rep=SCANVI_LATENT_KEY)
sc.tl.umap(adata_snraw, min_dist=0.3)

In [ ]:
sc.pl.umap(
    adata_snraw,
    color=["cell_type"],
    frameon=False,
    ncols=1,
)

In [ ]:
adata_snraw.write("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/snRNA_v1/snRNA_scanvi_01.h5ad")

In [ ]:
adata_snraw

In [ ]:
adata_snrna=sc.read("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/snRNA_v1/snRNA_scanvi_01.h5ad")

In [ ]:
adata_snrna.obs['cell_type'].value_counts()

In [ ]:
adata_snrna.obs['cell_type']